In [ ]:
# Tis Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Methodology: 2.5D Multiple-Instance Learning with Frozen Pretrained Backbones

## Overview

Instead of training a 3D encoder from scratch (slow, GPU-heavy), this pipeline reuses an existing pretrained 2D vision backbone as a frozen feature extractor, and only trains a small attention-pooling layer plus classification heads on top. This is fast, CPU/GPU-light, and avoids the multi-hour training runs of the SSL approach.

## Phase 1 — Data Foundation (already done, reused as-is)

**What it does:**
- Loads the metadata TSV, drops patients with missing IDH
- Verifies file completeness per patient (all 4 MRI sequences + segmentation mask actually exist on disk)
- Builds two cohorts: one using FeTS auto-segmentation (larger), one using manual segmentation only (smaller, higher quality)
- Audits final cohort: label distributions, missing values, class balance



In [ ]:
import os
import pandas as pd
from collections import Counter

# ============================================================
# PHASE 1 — DATA FOUNDATION
# Loads metadata, drops missing-IDH patients, verifies file
# completeness, builds FeTS and manual-segmentation cohorts,
# and audits the final dataset.
# ============================================================

metadata_path = "/kaggle/input/datasets/tano1974/glioma/UTSW_Glioma_Metadata-2-1.tsv"
base_path = "/kaggle/input/datasets/tano1974/glioma/PKG - UTSW-Glioma/PKG - UTSW-Glioma/UTSW-Glioma"

# ------------------------------------------------------------
# Step 1: Load metadata, drop missing IDH
# ------------------------------------------------------------
df = pd.read_csv(metadata_path, sep="\t")
print("Original metadata shape:", df.shape)

df = df.dropna(subset=["IDH"]).copy()
df["IDH_Label"] = df["IDH"].map({
    "wild type": 0,
    "mutated": 1
})

print("\nAfter removing missing IDH:", df.shape)
print("\nIDH Distribution:")
print(df["IDH_Label"].value_counts())

# ------------------------------------------------------------
# Step 2: Verify file completeness per patient (FeTS cohort)
# ------------------------------------------------------------
required_files = {
    "brain_t1_ants.nii",
    "brain_t1ce_ants.nii",
    "brain_t2_ants.nii",
    "brain_fl_ants.nii",
    "tumorseg_FeTS.nii"
}

complete_patients = []
missing_patients = []

for patient in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient)
    if not os.path.isdir(patient_path):
        continue
    files = set(os.listdir(patient_path))
    if required_files.issubset(files):
        complete_patients.append(patient)
    else:
        missing_patients.append(patient)

print("\nPatients with complete MRI data (FeTS):", len(complete_patients))
print("Patients with missing data:", len(missing_patients))

# ------------------------------------------------------------
# Step 3: Build the FeTS master dataset
# ------------------------------------------------------------
master_df = df[df["Subject ID"].isin(complete_patients)].copy()

master_df = master_df[[
    "Subject ID",
    "IDH",
    "IDH_Label",
    "Age at Imaging",
    "Sex at birth",
    "Tumor Grade",
    "Tumor Type",
    "Scanner Make",
    "Scanner Model",
    "Scanner Strength"
]]

master_df.rename(columns={
    "Subject ID": "PatientID",
    "Age at Imaging": "Age",
    "Sex at birth": "Sex",
    "Tumor Grade": "Grade"
}, inplace=True)

master_df["T1_Path"] = master_df["PatientID"].apply(
    lambda x: os.path.join(base_path, x, "brain_t1_ants.nii")
)
master_df["T1CE_Path"] = master_df["PatientID"].apply(
    lambda x: os.path.join(base_path, x, "brain_t1ce_ants.nii")
)
master_df["T2_Path"] = master_df["PatientID"].apply(
    lambda x: os.path.join(base_path, x, "brain_t2_ants.nii")
)
master_df["FLAIR_Path"] = master_df["PatientID"].apply(
    lambda x: os.path.join(base_path, x, "brain_fl_ants.nii")
)
master_df["Mask_Path"] = master_df["PatientID"].apply(
    lambda x: os.path.join(base_path, x, "tumorseg_FeTS.nii")
)

path_columns = ["T1_Path", "T1CE_Path", "T2_Path", "FLAIR_Path", "Mask_Path"]

for col in path_columns:
    master_df[col + "_Exists"] = master_df[col].apply(os.path.exists)

print("\nPath Verification")
for col in path_columns:
    print(col, ":", master_df[col + "_Exists"].sum(), "/", len(master_df))

exist_cols = [col + "_Exists" for col in path_columns]
master_df = master_df[master_df[exist_cols].all(axis=1)].copy()
master_df.drop(columns=exist_cols, inplace=True)

print("\nFinal FeTS Master Dataset Shape:", master_df.shape)
print("\nFinal IDH Distribution:")
print(master_df["IDH_Label"].value_counts())
print("\nFinal IDH Percentage:")
print(master_df["IDH_Label"].value_counts(normalize=True) * 100)

master_df.to_csv("/kaggle/working/master_dataset_fets.csv", index=False)
print("\nSaved: /kaggle/working/master_dataset_fets.csv")

# ------------------------------------------------------------
# Step 4: Build the manual-segmentation subset (higher quality)
# ------------------------------------------------------------
required_manual = {
    "brain_t1_ants.nii",
    "brain_t1ce_ants.nii",
    "brain_t2_ants.nii",
    "brain_fl_ants.nii",
    "tumorseg_manual_correction.nii"
}

manual_patients = []

for patient in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient)
    if not os.path.isdir(patient_path):
        continue
    files = set(os.listdir(patient_path))
    if required_manual.issubset(files):
        manual_patients.append(patient)

manual_df = df[df["Subject ID"].isin(manual_patients)].copy()

manual_df = manual_df[[
    "Subject ID",
    "IDH",
    "IDH_Label",
    "Age at Imaging",
    "Sex at birth",
    "Tumor Grade"
]]

manual_df.rename(columns={
    "Subject ID": "PatientID",
    "Age at Imaging": "Age",
    "Sex at birth": "Sex",
    "Tumor Grade": "Grade"
}, inplace=True)

manual_df.to_csv("/kaggle/working/master_dataset_manual_subset.csv", index=False)

print("\nManual Subset Shape:", manual_df.shape)
print("\nManual IDH Distribution:")
print(manual_df["IDH_Label"].value_counts())
print("\nSaved: /kaggle/working/master_dataset_manual_subset.csv")

# ------------------------------------------------------------
# Step 5: Audit the final FeTS cohort
# ------------------------------------------------------------
audit_df = pd.read_csv("/kaggle/working/master_dataset_fets.csv")

print("\n" + "="*60)
print("FINAL DATASET AUDIT (FeTS cohort)")
print("="*60)

print("\nDataset shape:", audit_df.shape)
print("\nIDH Distribution:")
print(audit_df["IDH_Label"].value_counts())
print("\nSex Distribution:")
print(audit_df["Sex"].value_counts())
print("\nGrade Distribution:")
print(audit_df["Grade"].value_counts())
print("\nAge Statistics:")
print(audit_df["Age"].describe())
print("\nTumour Type Distribution:")
print(audit_df["Tumor Type"].value_counts())
print("\nMissing Values:")
print(audit_df.isnull().sum())

print("\n" + "="*60)
print("PHASE 1 COMPLETE")
print("="*60)

## IDH mutation status by tumour grade

Loads master_dataset_fets.csv from phase 1, drops the 7 patients missing a grade value, builds a grade x IDH crosstab, and plots it as a stacked bar chart (grade 2/3/4 on x-axis, wild-type vs mutated counts stacked). Saves as png and pdf to /kaggle/working/.

Useful as a companion to the cohort table — shows IDH mutations skew toward lower grade tumours, which matches what's expected clinically and is a quick sanity check that the cohort looks right before moving on to modelling.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "figure.dpi": 300,
})

master_df = pd.read_csv("/kaggle/working/master_dataset_fets.csv")

# Drop the 7 patients with missing Grade before plotting
plot_df = master_df.dropna(subset=["Grade"]).copy()
plot_df["Grade"] = plot_df["Grade"].astype(int)

crosstab = pd.crosstab(plot_df["Grade"], plot_df["IDH_Label"])
crosstab.columns = ["Wild type", "Mutated"]

print("Actual crosstab (Grade x IDH):")
print(crosstab)

fig, ax = plt.subplots(figsize=(4.5, 3.2))
crosstab.plot(kind="bar", stacked=True, ax=ax, color=["#c9782f", "#1f6f8b"], width=0.65)

ax.set_xlabel("Tumour Grade")
ax.set_ylabel("Number of Patients")
ax.set_title("IDH Mutation Status Across Tumour Grades", fontsize=9.5)
ax.legend(title="IDH Status", fontsize=8, title_fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xticks(rotation=0)

fig.tight_layout()
fig.savefig("/kaggle/working/idh_by_grade.png", dpi=300, bbox_inches="tight")
fig.savefig("/kaggle/working/idh_by_grade.pdf", bbox_inches="tight")
plt.show()

print("\nSaved: idh_by_grade.png and idh_by_grade.pdf")

## Phase 2 — MRI Preprocessing & Tumour-Guided Slice Extraction

**What it does:**
- For each patient, normalize each MRI sequence (z-score within brain voxels, clip outliers)
- Using the segmentation mask, identify the axial slice range that actually contains tumour tissue
- Extract a fixed number of slices (e.g., 10-20) spanning that tumour region, not the whole brain, not random slices
- For each extracted slice, stack all 4 sequences (T1, T1CE, T2, FLAIR) as channels, producing one multi-channel 2D image per slice
- Result: each patient becomes a bag of K tumour-region slices, rather than one 3D volume

**Why this matters:** this is what makes the next phase possible, 2D pretrained backbones expect 2D images, not 3D volumes, so this conversion step bridges the two.



In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom

NUM_SLICES = 15
SLICE_SIZE = (224, 224)
MARGIN = 2
EPS = 1e-6

OUT_DIR = "/kaggle/working/slices"
os.makedirs(OUT_DIR, exist_ok=True)

master_df = pd.read_csv("/kaggle/working/master_dataset_fets.csv")


def load_nii(path):
    return np.asarray(nib.load(path).get_fdata(), dtype=np.float32)


def zscore(vol, mask):
    voxels = vol[mask]
    if voxels.size == 0:
        return vol
    mean = voxels.mean()
    std = voxels.std()
    return np.clip((vol - mean) / (std + EPS), -5, 5)


def tumour_z_range(seg, margin):
    present = np.any(seg > 0, axis=(0, 1))
    idx = np.nonzero(present)[0]
    if idx.size == 0:
        return None
    z_min = max(0, int(idx.min()) - margin)
    z_max = min(seg.shape[2] - 1, int(idx.max()) + margin)
    return z_min, z_max


def resize2d(img, size):
    factors = (size[0] / img.shape[0], size[1] / img.shape[1])
    return zoom(img, factors, order=1).astype(np.float32)


def extract_patient(row):
    t1 = load_nii(row["T1_Path"])
    t1ce = load_nii(row["T1CE_Path"])
    t2 = load_nii(row["T2_Path"])
    flair = load_nii(row["FLAIR_Path"])
    seg = load_nii(row["Mask_Path"])

    z_range = tumour_z_range(seg, MARGIN)
    if z_range is None:
        return None
    z_min, z_max = z_range

    brain_mask = (t1 > 0) | (t1ce > 0) | (t2 > 0) | (flair > 0)
    t1 = zscore(t1, brain_mask)
    t1ce = zscore(t1ce, brain_mask)
    t2 = zscore(t2, brain_mask)
    flair = zscore(flair, brain_mask)

    if z_max == z_min:
        z_indices = [z_min] * NUM_SLICES
    else:
        z_indices = np.round(np.linspace(z_min, z_max, NUM_SLICES)).astype(int).tolist()

    bag = np.zeros((NUM_SLICES, 4, *SLICE_SIZE), dtype=np.float32)
    for i, z in enumerate(z_indices):
        bag[i, 0] = resize2d(t1[:, :, z], SLICE_SIZE)
        bag[i, 1] = resize2d(t1ce[:, :, z], SLICE_SIZE)
        bag[i, 2] = resize2d(t2[:, :, z], SLICE_SIZE)
        bag[i, 3] = resize2d(flair[:, :, z], SLICE_SIZE)

    return bag


processed = []
skipped = []

for _, row in master_df.iterrows():
    pid = row["PatientID"]
    bag = extract_patient(row)
    if bag is None:
        skipped.append(pid)
        continue
    np.savez_compressed(os.path.join(OUT_DIR, f"{pid}.npz"), slices=bag)
    processed.append(pid)

    if len(processed) % 50 == 0:
        print(f"{len(processed)} patients done")

print(f"\nDone. Processed: {len(processed)}, skipped: {len(skipped)}")
if skipped:
    print("Skipped (empty mask):", skipped[:20])

master_df["has_slices"] = master_df["PatientID"].isin(processed)
master_df.to_csv("/kaggle/working/master_dataset_fets_with_slices.csv", index=False)
print("Saved: /kaggle/working/master_dataset_fets_with_slices.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "figure.dpi": 300,
})

PATIENT_ID = "BT0001"
SLICE_INDICES = [0, 7, 14]
SEQUENCE_NAMES = ["T1", "T1CE", "T2", "FLAIR"]

bag = np.load(f"/kaggle/working/slices/{PATIENT_ID}.npz")["slices"]

fig, axes = plt.subplots(len(SLICE_INDICES), 4, figsize=(8, 6))

for row, slice_idx in enumerate(SLICE_INDICES):
    for col in range(4):
        ax = axes[row, col]
        img = bag[slice_idx, col]
        ax.imshow(img, cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(SEQUENCE_NAMES[col], fontsize=9)
        if col == 0:
            ax.set_ylabel(f"Slice {slice_idx}", fontsize=9)

fig.suptitle(f"Example Tumour-Guided Slice Bag — Patient {PATIENT_ID}", fontsize=10)
fig.tight_layout()
fig.savefig("/kaggle/working/sample_slice_bag.png", dpi=300, bbox_inches="tight")
fig.savefig("/kaggle/working/sample_slice_bag.pdf", bbox_inches="tight")
plt.show()


## Phase 3 — Frozen Backbone Feature Extraction

**What it does:**
- Pass every extracted slice through a pretrained 2D vision backbone (e.g., ConvNeXt or EfficientNetV2, pretrained on ImageNet)
- The backbone weights are frozen, no training, no backpropagation through it, just a forward pass to get a feature embedding per slice
- Save the resulting embeddings (one fixed-length vector per slice, per patient)

**Why this is fast:** no gradient computation, no optimizer step, just inference.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models

SLICES_DIR = "/kaggle/working/slices"
OUT_DIR = "/kaggle/working/features"
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

master_df = pd.read_csv("/kaggle/working/master_dataset_fets_with_slices.csv")
master_df = master_df[master_df["has_slices"] == True]

# ---- Build backbone, adapt first conv layer for 4-channel MRI input ----
backbone = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)

old_conv = backbone.features[0][0]
new_conv = nn.Conv2d(
    in_channels=4,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None,
)

with torch.no_grad():
    new_conv.weight[:, :3] = old_conv.weight
    new_conv.weight[:, 3] = old_conv.weight.mean(dim=1)

backbone.features[0][0] = new_conv
backbone.classifier = nn.Identity()

for p in backbone.parameters():
    p.requires_grad = False

backbone.eval()
backbone.to(device)

EMBED_DIM = 1280

processed = []

with torch.no_grad():
    for _, row in master_df.iterrows():
        pid = row["PatientID"]
        bag = np.load(os.path.join(SLICES_DIR, f"{pid}.npz"))["slices"]
        bag_t = torch.from_numpy(bag).float().to(device)

        embeddings = []
        for i in range(0, bag_t.shape[0], BATCH_SIZE):
            batch = bag_t[i:i + BATCH_SIZE]
            out = backbone(batch)
            embeddings.append(out.cpu().numpy())

        patient_embeddings = np.concatenate(embeddings, axis=0)
        np.save(os.path.join(OUT_DIR, f"{pid}.npy"), patient_embeddings)
        processed.append(pid)

        if len(processed) % 50 == 0:
            print(f"{len(processed)} patients done")

print(f"\nDone. Extracted features for {len(processed)} patients.")
print(f"Each patient: (num_slices, {EMBED_DIM}) embedding array saved to {OUT_DIR}")

master_df["has_features"] = master_df["PatientID"].isin(processed)
master_df.to_csv("/kaggle/working/master_dataset_fets_with_features.csv", index=False)
print("Saved: /kaggle/working/master_dataset_fets_with_features.csv")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.manifold import TSNE

matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "figure.dpi": 300,
})

FEATURES_DIR = "/kaggle/working/features"

master_df = pd.read_csv("/kaggle/working/master_dataset_fets_with_features.csv")
master_df = master_df[master_df["has_features"] == True]

pooled = []
labels = []
patient_ids = []

for _, row in master_df.iterrows():
    pid = row["PatientID"]
    slice_embeddings = np.load(os.path.join(FEATURES_DIR, f"{pid}.npy"))
    patient_vector = slice_embeddings.mean(axis=0)
    pooled.append(patient_vector)
    labels.append(row["IDH_Label"])
    patient_ids.append(pid)

X = np.stack(pooled)
y = np.array(labels)

print(f"Pooled feature matrix shape: {X.shape}")
print(f"Patients: {len(y)}, IDH wild-type: {(y == 0).sum()}, IDH mutated: {(y == 1).sum()}")

tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca")
X_2d = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(5, 4.2))

for label_val, label_name, color in [(0, "Wild type", "#c9782f"), (1, "Mutated", "#1f6f8b")]:
    mask = y == label_val
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=color, label=f"IDH {label_name} (n={mask.sum()})",
        s=18, alpha=0.75, edgecolors="none",
    )

ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.set_title("Patient-Level Pooled Backbone Embeddings by IDH Status", fontsize=9.5)
ax.legend(fontsize=8, loc="best")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig("/kaggle/working/tsne_idh_embeddings.png", dpi=300, bbox_inches="tight")
fig.savefig("/kaggle/working/tsne_idh_embeddings.pdf", bbox_inches="tight")
plt.show()

print("Saved: tsne_idh_embeddings.png and tsne_idh_embeddings.pdf")

## Phase 4 — Attention-Based MIL Pooling and Multi-Task Training (the only actual training)

**What it does:**
- For each patient, take their bag of slice embeddings from Phase 3
- Pass them through a small trainable attention pooling network, this learns to weight each slice by how informative it is, then combines them into one patient-level representation
- Feed that pooled representation into separate lightweight classification heads for IDH, Tumor Grade, MGMT, and 1p/19Q
- Train using a masked multi-task loss, patients missing a label for a given task simply do not contribute to that task's loss, so the whole cohort stays usable even with partial labels

**Why this is fast:** the only thing being trained is a small attention layer plus a few small classifier heads. Training operates on cached feature vectors, not raw images.




In [ ]:
import os
import subprocess
import urllib.request
import json

try:
    import radiomics
    print(f"Already installed: {radiomics.__version__}")
except ImportError:
    with urllib.request.urlopen('https://pypi.org/pypi/pyradiomics/3.0.1/json') as r:
        pypi_data = json.load(r)
    tar_url = next(f['url'] for f in pypi_data['urls'] if f['filename'].endswith('.tar.gz'))

    build_dir = '/kaggle/working/pyrad_build'
    os.makedirs(build_dir, exist_ok=True)
    orig_cwd = os.getcwd()
    os.chdir(build_dir)

    subprocess.run(['curl', '-sL', '-o', 'pyradiomics.tar.gz', tar_url], check=True)
    subprocess.run(['tar', 'xzf', 'pyradiomics.tar.gz'], check=True)
    src_dir = [d for d in os.listdir('.') if d.startswith('pyradiomics-')][0]
    versioneer_path = os.path.join(src_dir, 'versioneer.py')

    with open(versioneer_path) as f:
        content = f.read()
    content = content.replace('configparser.SafeConfigParser()', 'configparser.ConfigParser()')
    content = content.replace('parser.readfp(f)', 'parser.read_file(f)')
    with open(versioneer_path, 'w') as f:
        f.write(content)

    subprocess.run(['pip', 'install', 'SimpleITK', 'PyWavelets', 'pykwalify', 'six'], check=True)
    result = subprocess.run(['pip', 'install', '.', '--no-build-isolation'],
                             cwd=src_dir, capture_output=True, text=True)
    print(result.stdout[-1000:])
    if result.returncode != 0:
        print("INSTALL FAILED:")
        print(result.stderr[-1500:])
    os.chdir(orig_cwd)

    import radiomics
    print(f"SUCCESS: pyradiomics {radiomics.__version__} installed")

In [ ]:
import os
print(sorted(os.listdir("/kaggle/working/")))

In [ ]:
import pandas as pd
import nibabel as nib
import numpy as np

master_df = pd.read_csv("/kaggle/working/master_dataset_fets_with_slices.csv")
row = master_df[master_df["PatientID"] == "BT0020"].iloc[0]

mask = nib.load(row["Mask_Path"]).get_fdata()
print("Mask shape:", mask.shape)
print("Tumor voxel count:", int((mask > 0).sum()))
print("Tumor volume as % of total:", 100 * (mask > 0).sum() / mask.size)

t1ce = nib.load(row["T1CE_Path"]).get_fdata()
print("T1CE shape:", t1ce.shape)
print("T1CE value range:", t1ce.min(), t1ce.max())

In [ ]:
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor
import logging
import time

logging.getLogger("radiomics").setLevel(logging.ERROR)

master_df = pd.read_csv("/kaggle/working/master_dataset_fets_with_slices.csv")
row = master_df[master_df["PatientID"] == "BT0020"].iloc[0]

settings = {"interpolator": None, "resampledPixelSpacing": None, "binCount": 32}
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
extractor.disableAllFeatures()
for fc in ["firstorder", "shape", "glcm", "glrlm", "glszm", "gldm"]:
    extractor.enableFeatureClassByName(fc)

mask_img = sitk.ReadImage(row["Mask_Path"])
image = sitk.ReadImage(row["T1CE_Path"])

start = time.time()
result = extractor.execute(image, mask_img, label=1)
print(f"BT0020 extraction completed in {time.time()-start:.1f} seconds")

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor
import logging

logging.getLogger("radiomics").setLevel(logging.ERROR)

MASTER_PATH = "/kaggle/working/master_dataset_fets_with_slices.csv"
OUT_PATH = "/kaggle/working/radiomics_features.csv"
SEQUENCES = {"T1CE": "T1CE_Path", "FLAIR": "FLAIR_Path"}

master_df = pd.read_csv(MASTER_PATH)
master_df = master_df[master_df["has_slices"] == True].reset_index(drop=True)

if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH)
    done_ids = set(existing_df["PatientID"])
    print(f"Resuming: {len(done_ids)} patients already extracted, skipping them.")
else:
    done_ids = set()

# Fixed: binCount instead of binWidth. binWidth=64 caused a ~12,000-bin
# explosion on patients with extreme outlier intensity values (e.g.
# BT0020's max of 765,553), since MRI intensities aren't standardized
# across scanners the way CT Hounsfield units are. binCount fixes the
# number of bins regardless of intensity range, avoiding this entirely.
settings = {"interpolator": None, "resampledPixelSpacing": None, "binCount": 32}
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
extractor.disableAllFeatures()
for fc in ["firstorder", "shape", "glcm", "glrlm", "glszm", "gldm"]:
    extractor.enableFeatureClassByName(fc)

remaining_df = master_df[~master_df["PatientID"].isin(done_ids)].reset_index(drop=True)
print(f"Extracting radiomics for {len(remaining_df)} remaining patients...")

failed = []

for i, row in remaining_df.iterrows():
    pid = row["PatientID"]
    print(f"[{i+1}/{len(remaining_df)}] Starting {pid}...", flush=True)

    patient_features = {"PatientID": pid}

    try:
        mask_img = sitk.ReadImage(row["Mask_Path"])
        mask_arr = (sitk.GetArrayFromImage(mask_img) > 0).astype(np.uint8)
        mask_img_binary = sitk.GetImageFromArray(mask_arr)
        mask_img_binary.CopyInformation(mask_img)

        for seq_name, path_col in SEQUENCES.items():
            image = sitk.ReadImage(row[path_col])
            result = extractor.execute(image, mask_img_binary, label=1)
            for key, value in result.items():
                if key.startswith("diagnostics"):
                    continue
                patient_features[f"{seq_name}_{key}"] = float(value)
            del image, result

        row_df = pd.DataFrame([patient_features])
        if os.path.exists(OUT_PATH):
            row_df.to_csv(OUT_PATH, mode="a", header=False, index=False)
        else:
            row_df.to_csv(OUT_PATH, index=False)

        print(f"[{i+1}/{len(remaining_df)}] Done {pid} (saved)", flush=True)

    except Exception as e:
        print(f"FAILED for {pid}: {e}", flush=True)
        failed.append(pid)

    del mask_img, mask_arr, mask_img_binary
    gc.collect()

    if (i + 1) % 50 == 0:
        print(f"--- Progress: {i+1}/{len(remaining_df)} ---", flush=True)

final_df = pd.read_csv(OUT_PATH)
print(f"\nDone. Total patients in {OUT_PATH}: {len(final_df)}")
print(f"Feature columns: {final_df.shape[1] - 1}")
if failed:
    print(f"Failed this run: {len(failed)} --", failed[:20])

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix, average_precision_score,
)
from sklearn.utils.class_weight import compute_class_weight

FEATURES_DIR = "/kaggle/working/features"
METADATA_PATH = "/kaggle/input/datasets/tano1974/glioma/UTSW_Glioma_Metadata-2-1.tsv"
MASTER_PATH = "/kaggle/working/master_dataset_fets_with_slices.csv"
RADIOMICS_PATH = "/kaggle/working/radiomics_features.csv"
CACHED_CLINICAL_PATH = "/kaggle/working/clinical_features_cache.csv"

EMBED_DIM = 1280
ATTN_HIDDEN = 128
DROPOUT_P = 0.4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 40       # reduced from 60
PATIENCE = 5          # reduced from 8
BATCH_SIZE = 16
LR = 1e-3
N_FOLDS = 5
N_BOOTSTRAP = 300      # reduced from 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: GPU not detected -- training will be much slower. "
          "Check Settings -> Accelerator in Kaggle.")

GRADE_MAP = {2.0: 0, 3.0: 1, 4.0: 2}
BINARY_MAP = {
    "MGMT": {"unmethylated": 0, "methylated": 1},
    "1p19Q CODEL": {"non co-deleted": 0, "co-deleted": 1},
}
TASKS = {
    "idh": {"num_classes": 2}, "grade": {"num_classes": 3},
    "mgmt": {"num_classes": 2}, "codel": {"num_classes": 2},
}

# ============================================================
# STEP 2: Clinical features (CACHED -- only computed once ever)
# ============================================================
print("\n" + "=" * 70 + "\nSTEP 2: Clinical + Radiomics Merge\n" + "=" * 70)


def load_nii(path):
    return np.asarray(nib.load(path).get_fdata(), dtype=np.float32)


master_df = pd.read_csv(MASTER_PATH)
master_df = master_df[master_df["has_slices"] == True].reset_index(drop=True)

if os.path.exists(CACHED_CLINICAL_PATH):
    print(f"Loading cached clinical features from {CACHED_CLINICAL_PATH} (skipping recomputation)")
    cache_df = pd.read_csv(CACHED_CLINICAL_PATH)
    master_df = master_df.merge(cache_df, on="PatientID", how="left")
else:
    print("No cache found -- computing clinical features (one-time cost)...")
    volumes, t1ce_means, flair_means = [], [], []
    for i, row in master_df.iterrows():
        seg = load_nii(row["Mask_Path"])
        mask = seg > 0
        volumes.append(int(mask.sum()))
        t1ce = load_nii(row["T1CE_Path"])
        flair = load_nii(row["FLAIR_Path"])
        t1ce_means.append(float(t1ce[mask].mean()) if mask.sum() > 0 else 0.0)
        flair_means.append(float(flair[mask].mean()) if mask.sum() > 0 else 0.0)
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(master_df)} patients processed")

    master_df["TumourVolumeVoxels"] = volumes
    master_df["T1CEMeanIntensity"] = t1ce_means
    master_df["FLAIRMeanIntensity"] = flair_means

    cache_out = master_df[["PatientID", "TumourVolumeVoxels", "T1CEMeanIntensity", "FLAIRMeanIntensity"]]
    cache_out.to_csv(CACHED_CLINICAL_PATH, index=False)
    print(f"Cached to {CACHED_CLINICAL_PATH} -- future runs will skip this step")

master_df["SexEncoded"] = master_df["Sex"].map({"M": 0, "F": 1})

base_clinical_cols = ["AgeNorm", "SexEncoded", "VolumeNorm", "T1CENorm", "FLAIRNorm"]
for col, out_col in [("Age", "AgeNorm"), ("TumourVolumeVoxels", "VolumeNorm"),
                      ("T1CEMeanIntensity", "T1CENorm"), ("FLAIRMeanIntensity", "FLAIRNorm")]:
    master_df[out_col] = (master_df[col] - master_df[col].mean()) / (master_df[col].std() + 1e-8)

radiomics_cols = []
if os.path.exists(RADIOMICS_PATH):
    radiomics_df = pd.read_csv(RADIOMICS_PATH)
    master_df = master_df.merge(radiomics_df, on="PatientID", how="left")
    radiomics_cols = [c for c in radiomics_df.columns if c != "PatientID"]
    for col in radiomics_cols:
        mean, std = master_df[col].mean(), master_df[col].std()
        master_df[col] = (master_df[col] - mean) / (std + 1e-8)
        master_df[col] = master_df[col].fillna(0.0)
    print(f"Merged {len(radiomics_cols)} full radiomics features")
else:
    print("No radiomics file -- using lightweight clinical features only")

CLINICAL_COLS = base_clinical_cols + radiomics_cols
CLINICAL_DIM = len(CLINICAL_COLS)
print(f"Total clinical/radiomic feature dimension: {CLINICAL_DIM}")

# ============================================================
# STEP 3: Merge labels
# ============================================================
raw_meta = pd.read_csv(METADATA_PATH, sep="\t")
raw_meta = raw_meta.rename(columns={"Subject ID": "PatientID"})
if "MGMT" not in master_df.columns:
    master_df = master_df.merge(raw_meta[["PatientID", "MGMT", "1p19Q CODEL"]], on="PatientID", how="left")


def encode_labels(row):
    out = {}
    out["idh_label"] = int(row["IDH_Label"])
    out["idh_valid"] = True
    grade_val = row.get("Grade", np.nan)
    if grade_val in GRADE_MAP:
        out["grade_label"] = GRADE_MAP[grade_val]
        out["grade_valid"] = True
    else:
        out["grade_label"] = 0
        out["grade_valid"] = False
    mgmt_val = row.get("MGMT", None)
    if isinstance(mgmt_val, str) and mgmt_val.strip() in BINARY_MAP["MGMT"]:
        out["mgmt_label"] = BINARY_MAP["MGMT"][mgmt_val.strip()]
        out["mgmt_valid"] = True
    else:
        out["mgmt_label"] = 0
        out["mgmt_valid"] = False
    codel_val = row.get("1p19Q CODEL", None)
    is_idh_mutant = int(row["IDH_Label"]) == 1
    if is_idh_mutant and isinstance(codel_val, str) and codel_val.strip() in BINARY_MAP["1p19Q CODEL"]:
        out["codel_label"] = BINARY_MAP["1p19Q CODEL"][codel_val.strip()]
        out["codel_valid"] = True
    else:
        out["codel_label"] = 0
        out["codel_valid"] = False
    return pd.Series(out)


labels_df = master_df.apply(encode_labels, axis=1)
master_df = pd.concat([master_df, labels_df], axis=1)

print("\nTask label availability:")
for task in TASKS:
    print(f"  {task}: {master_df[f'{task}_valid'].sum()} / {len(master_df)} valid")


# ============================================================
# STEP 4: Dataset + Model
# ============================================================
class GliomaFeatureDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["PatientID"]
        embeddings = np.load(os.path.join(FEATURES_DIR, f"{pid}.npy"))
        clinical = row[CLINICAL_COLS].values.astype(np.float32)
        sample = {"embeddings": torch.from_numpy(embeddings).float(),
                  "clinical": torch.from_numpy(clinical).float(), "PatientID": pid}
        for task in TASKS:
            sample[f"{task}_label"] = torch.tensor(row[f"{task}_label"], dtype=torch.long)
            sample[f"{task}_valid"] = torch.tensor(bool(row[f"{task}_valid"]), dtype=torch.bool)
        return sample


class AttentionMIL(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout_p):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.Tanh(),
                                        nn.Dropout(dropout_p), nn.Linear(hidden_dim, 1))

    def forward(self, x):
        w = torch.softmax(self.attention(x), dim=1)
        return (x * w).sum(dim=1), w.squeeze(-1)


class MultiTaskMILModel(nn.Module):
    def __init__(self, embed_dim, clinical_dim, attn_hidden, tasks, dropout_p):
        super().__init__()
        self.pooling = AttentionMIL(embed_dim, attn_hidden, dropout_p)
        self.fusion_dropout = nn.Dropout(dropout_p)
        fused_dim = embed_dim + clinical_dim
        self.heads = nn.ModuleDict({
            t: nn.Sequential(nn.Linear(fused_dim, 64), nn.ReLU(),
                              nn.Dropout(dropout_p), nn.Linear(64, c["num_classes"]))
            for t, c in tasks.items()
        })

    def forward(self, x, clinical):
        pooled, attn = self.pooling(x)
        fused = self.fusion_dropout(torch.cat([pooled, clinical], dim=1))
        return {t: h(fused) for t, h in self.heads.items()}, attn


def masked_multitask_loss(logits, batch, tasks, class_weights):
    total = 0.0
    for t in tasks:
        valid = batch[f"{t}_valid"]
        if valid.sum() == 0:
            continue
        loss = nn.functional.cross_entropy(logits[t][valid], batch[f"{t}_label"][valid],
                                            weight=class_weights.get(t))
        total = total + loss
    return total


def compute_class_weights(train_df, tasks):
    cw = {}
    for task, cfg in tasks.items():
        valid_mask = train_df[f"{task}_valid"]
        if valid_mask.sum() < 2:
            continue
        labels = train_df.loc[valid_mask, f"{task}_label"].values
        present = np.unique(labels)
        if len(present) < 2:
            continue
        w = compute_class_weight("balanced", classes=present, y=labels)
        full_w = np.ones(cfg["num_classes"], dtype=np.float32)
        for cls, ww in zip(present, w):
            full_w[int(cls)] = ww
        cw[task] = torch.tensor(full_w, dtype=torch.float32).to(device)
    return cw


# ============================================================
# STEP 5: 5-fold cross-validation
# ============================================================
print("\n" + "=" * 70 + "\nSTEP 5: 5-Fold Cross-Validation\n" + "=" * 70)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_logits = {t: {} for t in TASKS}
oof_labels = {}
oof_valid = {t: {} for t in TASKS}
fold_idh_aucs = []

for fold_idx, (train_val_idx, test_idx) in enumerate(skf.split(master_df, master_df["idh_label"])):
    print(f"\n{'='*70}\nFOLD {fold_idx+1}/{N_FOLDS}\n{'='*70}")
    train_val_df = master_df.iloc[train_val_idx].reset_index(drop=True)
    test_df = master_df.iloc[test_idx].reset_index(drop=True)
    train_df, val_df = train_test_split(train_val_df, test_size=0.15,
                                          stratify=train_val_df["idh_label"], random_state=fold_idx)

    class_weights = compute_class_weights(train_df, TASKS)
    train_loader = DataLoader(GliomaFeatureDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(GliomaFeatureDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(GliomaFeatureDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)

    model = MultiTaskMILModel(EMBED_DIM, CLINICAL_DIM, ATTN_HIDDEN, TASKS, DROPOUT_P).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        for batch in train_loader:
            emb = batch["embeddings"].to(device)
            clin = batch["clinical"].to(device)
            b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
            optimizer.zero_grad()
            logits, _ = model(emb, clin)
            loss = masked_multitask_loss(logits, b, TASKS, class_weights)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                emb = batch["embeddings"].to(device)
                clin = batch["clinical"].to(device)
                b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                logits, _ = model(emb, clin)
                val_loss += masked_multitask_loss(logits, b, TASKS, class_weights).item()
        val_loss /= len(val_loader)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f"/kaggle/working/mil_model_fold{fold_idx}.pt")
    print(f"Fold {fold_idx+1} best val_loss: {best_val_loss:.4f} (model saved)")

    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            emb = batch["embeddings"].to(device)
            clin = batch["clinical"].to(device)
            logits, _ = model(emb, clin)
            pids = batch["PatientID"]
            for i, pid in enumerate(pids):
                oof_labels[pid] = {t: batch[f"{t}_label"][i].item() for t in TASKS}
                for t in TASKS:
                    oof_logits[t][pid] = logits[t][i].cpu().numpy()
                    oof_valid[t][pid] = batch[f"{t}_valid"][i].item()

    idh_probs, idh_true = [], []
    for pid in test_df["PatientID"]:
        if oof_valid["idh"][pid]:
            p = torch.softmax(torch.tensor(oof_logits["idh"][pid]), dim=0).numpy()
            idh_probs.append(p[1])
            idh_true.append(oof_labels[pid]["idh"])
    if len(set(idh_true)) > 1:
        fold_auc = roc_auc_score(idh_true, idh_probs)
        fold_idh_aucs.append(fold_auc)
        print(f"Fold {fold_idx+1} IDH AUC: {fold_auc:.3f}")

print(f"\n{'='*70}")
print(f"PER-FOLD IDH AUC: {[f'{a:.3f}' for a in fold_idh_aucs]}")
print(f"Mean ± SD: {np.mean(fold_idh_aucs):.3f} ± {np.std(fold_idh_aucs):.3f}")


# ============================================================
# STEP 6: Pooled out-of-fold metrics (HONEST RESULT)
# ============================================================
def bootstrap_auc_ci(y_true, probs, num_classes, n_boot=N_BOOTSTRAP, ci=95, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        pp = probs[idx]
        try:
            if num_classes == 2:
                boot_aucs.append(roc_auc_score(yt, pp[:, 1]))
            else:
                boot_aucs.append(roc_auc_score(yt, pp, multi_class="ovr"))
        except ValueError:
            continue
    if len(boot_aucs) == 0:
        return float("nan"), float("nan"), float("nan")
    return (np.mean(boot_aucs), np.percentile(boot_aucs, (100-ci)/2), np.percentile(boot_aucs, 100-(100-ci)/2))


print(f"\n{'='*70}\nPOOLED OUT-OF-FOLD PERFORMANCE (all {len(master_df)} patients) -- HONEST RESULT\n{'='*70}")

for task in TASKS:
    num_classes = TASKS[task]["num_classes"]
    pids = [p for p in master_df["PatientID"] if oof_valid[task].get(p, False)]
    print(f"\n{'-'*70}\n{task.upper()}  (n={len(pids)})\n{'-'*70}")
    if len(pids) < 2:
        print("  Insufficient valid samples.")
        continue
    yv = np.array([oof_labels[p][task] for p in pids])
    probs = np.stack([torch.softmax(torch.tensor(oof_logits[task][p]), dim=0).numpy() for p in pids])
    preds = probs.argmax(axis=1)
    if len(np.unique(yv)) < 2:
        print("  Only one class present, skipping.")
        continue

    acc = accuracy_score(yv, preds)
    bal_acc = balanced_accuracy_score(yv, preds)
    f1 = f1_score(yv, preds, average="macro")
    precision = precision_score(yv, preds, average="macro", zero_division=0)
    recall = recall_score(yv, preds, average="macro", zero_division=0)
    cm = confusion_matrix(yv, preds, labels=list(range(num_classes)))

    try:
        if num_classes == 2:
            auc = roc_auc_score(yv, probs[:, 1])
            pr_auc = average_precision_score(yv, probs[:, 1])
        else:
            auc = roc_auc_score(yv, probs, multi_class="ovr")
            pr_auc = float("nan")
    except ValueError:
        auc = float("nan")
        pr_auc = float("nan")

    mean_auc, ci_lo, ci_hi = bootstrap_auc_ci(yv, probs, num_classes)

    print(f"  Accuracy:            {acc:.3f}")
    print(f"  Balanced accuracy:   {bal_acc:.3f}")
    print(f"  Macro F1:            {f1:.3f}")
    print(f"  Macro precision:     {precision:.3f}")
    print(f"  Macro recall:        {recall:.3f}")
    print(f"  AUC-ROC:             {auc:.3f}  (bootstrap mean={mean_auc:.3f}, 95% CI: {ci_lo:.3f}-{ci_hi:.3f})")
    if num_classes == 2:
        print(f"  PR-AUC:              {pr_auc:.3f}")
    print(f"  Confusion matrix:\n  {cm}")

    for cls in range(num_classes):
        tp = cm[cls, cls]
        fn = cm[cls, :].sum() - tp
        fp = cm[:, cls].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
        print(f"    Class {cls}: sensitivity={sens:.3f}, specificity={spec:.3f}")

print(f"\n{'='*70}\nDONE\n{'='*70}")

In [ ]:
import os
print(os.path.exists("/kaggle/working/master_dataset_fets_with_slices.csv"))
print(os.path.exists("/kaggle/working/clinical_features_cache.csv"))
print(os.path.exists("/kaggle/working/radiomics_features.csv"))
print(os.path.exists("/kaggle/working/features"))

## Phase 5 — Baseline and Ablation Comparisons

Compare, under identical splits:
- (i) Prior fully-supervised 3D CNN baseline from the first paper
- (ii) 2.5D MIL, single-task
- (iii) 2.5D MIL, multi-task
- (iv) 2.5D MIL, multi-task plus clinical/radiomic feature fusion

**Purpose:** isolates exactly how much benefit comes from transfer learning versus multi-task learning versus fusion.


In [ ]:
# === PHASE 5 (REAL PIPELINE, w/ ETA) TRANSFER CELL -- paste whole cell, run once ===
import base64, hashlib

EXPECTED_SHA256 = "791a1e58c655867b4e767fe1905ff164d05b964f7b332ea2ccb4336a85bc28d5"
EXPECTED_SIZE = 27462

_B64 = (
    "..."
)

_data = base64.b64decode(_B64)
_actual_sha256 = hashlib.sha256(_data).hexdigest()

assert len(_data) == EXPECTED_SIZE, f"Size mismatch: got {len(_data)}, expected {EXPECTED_SIZE}"
assert _actual_sha256 == EXPECTED_SHA256, f"Checksum mismatch: got {_actual_sha256}, expected {EXPECTED_SHA256}"

with open("/kaggle/working/phase5_ablation_real.py", "wb") as f:
    f.write(_data)

print(f"OK -- wrote /kaggle/working/phase5_ablation_real.py ({len(_data)} bytes, checksum verified)")

In [ ]:
_phase5_globals = {"__name__": "phase5_module"}
exec(open("/kaggle/working/phase5_ablation_real.py").read(), _phase5_globals)
globals().update(_phase5_globals)

In [ ]:
def masked_multitask_loss_fixed(logits, batch, tasks, class_weights):
    total = None
    for t in tasks:
        valid = batch[f"{t}_valid"]
        if valid.sum() == 0:
            continue
        loss = nn.functional.cross_entropy(logits[t][valid], batch[f"{t}_label"][valid],
                                            weight=class_weights.get(t))
        total = loss if total is None else total + loss
    if total is None:
        # No valid labels for any task in this batch -- return a zero loss
        # still connected to the graph so .backward() doesn't crash,
        # but it contributes no gradient (harmless no-op step).
        any_task = next(iter(logits))
        total = logits[any_task].sum() * 0.0
    return total

_phase5_globals["masked_multitask_loss"] = masked_multitask_loss_fixed
print("Patched masked_multitask_loss -- safe to resume")

In [ ]:
all_results += run_mil_condition(
    master_df, clinical_cols, folds, tasks={"mgmt": TASKS["mgmt"]},
    use_fusion=False, condition_name="(ii) 2.5D MIL single-task [mgmt]"
)
all_results += run_mil_condition(
    master_df, clinical_cols, folds, tasks={"codel": TASKS["codel"]},
    use_fusion=False, condition_name="(ii) 2.5D MIL single-task [codel]"
)
all_results += run_mil_condition(
    master_df, clinical_cols, folds, tasks=TASKS,
    use_fusion=False, condition_name="(iii) 2.5D MIL multi-task"
)
all_results += run_mil_condition(
    master_df, clinical_cols, folds, tasks=TASKS,
    use_fusion=True, condition_name="(iv) 2.5D MIL multi-task + fusion"
)

In [ ]:
import time

cache_dir = "/kaggle/working/baseline_volumes"
os.makedirs(cache_dir, exist_ok=True)

all_pids = master_df["PatientID"].tolist()
already_cached = [pid for pid in all_pids if os.path.exists(os.path.join(cache_dir, f"{pid}.npy"))]
remaining = [pid for pid in all_pids if pid not in set(already_cached)]

print(f"{len(already_cached)} already cached, {len(remaining)} remaining")

pid_to_row = {row["PatientID"]: row for _, row in master_df.iterrows()}

start = time.time()
for i, pid in enumerate(remaining):
    row = pid_to_row[pid]
    vol = load_and_resample_volume(row)
    np.save(os.path.join(cache_dir, f"{pid}.npy"), vol)
    if (i + 1) % 20 == 0 or (i + 1) == len(remaining):
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        eta = (len(remaining) - (i + 1)) / rate if rate > 0 else float("nan")
        print(f"  {i+1}/{len(remaining)} cached  "
              f"({elapsed/60:.1f}min elapsed, {rate:.2f} patients/sec, ETA {eta/60:.1f}min)")

print("Volume caching complete.")

In [ ]:
baseline_results = run_baseline_condition(master_df, folds)
all_results += baseline_results

In [ ]:
print(len(all_results))
print([r["condition"] for r in all_results])

In [ ]:
seen = set()
deduped_results = []
for r in all_results:
    key = (r["condition"], r["task"])
    if key not in seen:
        seen.add(key)
        deduped_results.append(r)

all_results = deduped_results
print(len(all_results))
print([r["condition"] for r in all_results])

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)
df_display = df[["condition", "task", "n", "accuracy", "balanced_accuracy",
                  "macro_f1", "auc", "auc_ci_low", "auc_ci_high"]]
df_display.to_csv("/kaggle/working/phase5_ablation_results.csv", index=False)

with pd.option_context("display.max_rows", None, "display.width", 140):
    print(df_display.round(3).to_string(index=False))

## Phase 6 — Label-Efficiency Protocol

Repeat Phase 4 training using only 10%, 25%, 50%, and 100% of the available labelled training data.

**Why this is now cheap to do properly:** since each training run takes minutes, this can run 4x per task, even with multiple random seeds for robustness, without the time cost that made this expensive under the SSL approach.

**Output:** label-efficiency curves, the central evidence that frozen-backbone transfer learning needs less labelled data than training from scratch.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Self-contained safe loss (avoids the earlier stale-globals issue)
def masked_multitask_loss_safe(logits, batch, tasks, class_weights):
    total = None
    for t in tasks:
        valid = batch[f"{t}_valid"]
        if valid.sum() == 0:
            continue
        loss = nn.functional.cross_entropy(logits[t][valid], batch[f"{t}_label"][valid],
                                            weight=class_weights.get(t))
        total = loss if total is None else total + loss
    if total is None:
        any_task = next(iter(logits))
        total = logits[any_task].sum() * 0.0
    return total

LABEL_FRACTIONS = [0.10, 0.25, 0.50, 1.00]
label_efficiency_results = []

for frac in LABEL_FRACTIONS:
    condition_name = f"Phase6 label-fraction={frac:.2f}"
    print(f"\n{'='*70}\n{condition_name}\n{'='*70}")

    oof_logits = {t: {} for t in TASKS}
    oof_labels = {}
    oof_valid = {t: {} for t in TASKS}

    for fold_idx, fold in enumerate(folds):
        train_val_df = master_df[master_df["PatientID"].isin(fold["train_val"])].reset_index(drop=True)
        test_df = master_df[master_df["PatientID"].isin(fold["test"])].reset_index(drop=True)
        train_df, val_df = train_test_split(train_val_df, test_size=0.15,
                                             stratify=train_val_df["idh_label"], random_state=fold_idx)

        if frac < 1.0:
            train_df, _ = train_test_split(train_df, train_size=frac,
                                            stratify=train_df["idh_label"], random_state=fold_idx)

        class_weights = compute_class_weights(train_df, TASKS)
        train_loader = DataLoader(GliomaFeatureDataset(train_df, clinical_cols, TASKS),
                                   batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(GliomaFeatureDataset(val_df, clinical_cols, TASKS),
                                 batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(GliomaFeatureDataset(test_df, clinical_cols, TASKS),
                                  batch_size=BATCH_SIZE, shuffle=False)

        model = MILModel(EMBED_DIM, TASKS, clinical_dim=len(clinical_cols)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val_loss = float("inf")
        best_state = None
        no_improve = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for batch in train_loader:
                emb = batch["embeddings"].to(device)
                clin = batch["clinical"].to(device)
                b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                optimizer.zero_grad()
                logits, _ = model(emb, clin)
                loss = masked_multitask_loss_safe(logits, b, TASKS, class_weights)
                loss.backward()
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    emb = batch["embeddings"].to(device)
                    clin = batch["clinical"].to(device)
                    b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                    logits, _ = model(emb, clin)
                    val_loss += masked_multitask_loss_safe(logits, b, TASKS, class_weights).item()
            val_loss /= len(val_loader)

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                break

        model.load_state_dict(best_state)
        print(f"  Fold {fold_idx+1} (n_train={len(train_df)}) best val_loss: {best_val_loss:.4f}")

        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                emb = batch["embeddings"].to(device)
                clin = batch["clinical"].to(device)
                logits, _ = model(emb, clin)
                pids = batch["PatientID"]
                for i, pid in enumerate(pids):
                    oof_labels[pid] = {t: batch[f"{t}_label"][i].item() for t in TASKS}
                    for t in TASKS:
                        oof_logits[t][pid] = logits[t][i].cpu().numpy()
                        oof_valid[t][pid] = batch[f"{t}_valid"][i].item()

    all_pids = master_df["PatientID"].tolist()
    for t in TASKS:
        r = summarize_task(t, TASKS[t]["num_classes"], oof_labels, oof_logits, oof_valid, all_pids)
        if r is None:
            continue
        r["label_fraction"] = frac
        r["condition"] = condition_name
        label_efficiency_results.append(r)
        print(f"  {t.upper():8s} AUC={r['auc']:.3f}  Acc={r['accuracy']:.3f}  BalAcc={r['balanced_accuracy']:.3f}")

print("\nPhase 6 complete.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 300})

df6 = pd.DataFrame(label_efficiency_results)
df6_display = df6[["label_fraction", "task", "n", "accuracy", "balanced_accuracy",
                    "macro_f1", "auc", "auc_ci_low", "auc_ci_high"]]
df6_display.to_csv("/kaggle/working/phase6_label_efficiency_results.csv", index=False)

with pd.option_context("display.max_rows", None, "display.width", 140):
    print(df6_display.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 4))
colors = {"idh": "#1f6f8b", "grade": "#c9782f", "mgmt": "#6a4c93", "codel": "#8b8b8b"}
for task in TASKS:
    sub = df6_display[df6_display["task"] == task].sort_values("label_fraction")
    ax.plot(sub["label_fraction"] * 100, sub["auc"], marker="o", label=task.upper(), color=colors[task])

ax.set_xlabel("% of labelled training data used")
ax.set_ylabel("AUC-ROC (pooled out-of-fold)")
ax.set_title("Label-Efficiency: Frozen-Backbone MIL + Fusion", fontsize=9.5)
ax.legend(fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig("/kaggle/working/label_efficiency_curve.png", dpi=300, bbox_inches="tight")
fig.savefig("/kaggle/working/label_efficiency_curve.pdf", bbox_inches="tight")
plt.show()

print("Saved: phase6_label_efficiency_results.csv, label_efficiency_curve.png/.pdf")

## Phase 7 — Uncertainty Quantification

- Monte Carlo Dropout: enable dropout at inference time, run multiple forward passes per patient, use the spread of predictions as an uncertainty estimate
- Lightweight ensembling: train several independently-initialized attention-pooling plus classifier heads on the same frozen backbone features, analogous to deep ensembles but fast since feature extraction only happens once
- Calibration: reliability diagrams plus Expected Calibration Error (ECE)
- Report accuracy specifically on the subset of confidently predicted cases


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

MC_PASSES = 20  # stochastic forward passes per patient for MC dropout

phase7_oof_probs = {t: {} for t in TASKS}   # pid -> mean prob array across MC passes
phase7_oof_std = {t: {} for t in TASKS}     # pid -> std across MC passes (uncertainty)
phase7_oof_true = {t: {} for t in TASKS}

for fold_idx, fold in enumerate(folds):
    print(f"\n--- Fold {fold_idx+1}/{N_FOLDS} ---")
    train_val_df = master_df[master_df["PatientID"].isin(fold["train_val"])].reset_index(drop=True)
    test_df = master_df[master_df["PatientID"].isin(fold["test"])].reset_index(drop=True)
    train_df, val_df = train_test_split(train_val_df, test_size=0.15,
                                         stratify=train_val_df["idh_label"], random_state=fold_idx)

    class_weights = compute_class_weights(train_df, TASKS)
    train_loader = DataLoader(GliomaFeatureDataset(train_df, clinical_cols, TASKS),
                               batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(GliomaFeatureDataset(val_df, clinical_cols, TASKS),
                             batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(GliomaFeatureDataset(test_df, clinical_cols, TASKS),
                              batch_size=BATCH_SIZE, shuffle=False)

    model = MILModel(EMBED_DIM, TASKS, clinical_dim=len(clinical_cols)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        for batch in train_loader:
            emb = batch["embeddings"].to(device)
            clin = batch["clinical"].to(device)
            b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
            optimizer.zero_grad()
            logits, _ = model(emb, clin)
            loss = masked_multitask_loss_safe(logits, b, TASKS, class_weights)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                emb = batch["embeddings"].to(device)
                clin = batch["clinical"].to(device)
                b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                logits, _ = model(emb, clin)
                val_loss += masked_multitask_loss_safe(logits, b, TASKS, class_weights).item()
        val_loss /= len(val_loader)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    print(f"  best val_loss: {best_val_loss:.4f}")

    # MC Dropout inference: keep dropout active (model.train()) but no grad, no weight updates
    model.train()
    pid_order = []
    accum_probs = {t: [] for t in TASKS}
    true_labels = {t: [] for t in TASKS}

    with torch.no_grad():
        for batch in test_loader:
            emb = batch["embeddings"].to(device)
            clin = batch["clinical"].to(device)
            pids = batch["PatientID"]
            pid_order.extend(pids)
            for t in TASKS:
                true_labels[t].extend(batch[f"{t}_label"].numpy())

            pass_probs = {t: [] for t in TASKS}
            for _ in range(MC_PASSES):
                logits, _ = model(emb, clin)
                for t in TASKS:
                    probs = F.softmax(logits[t], dim=-1).cpu().numpy()
                    pass_probs[t].append(probs)
            for t in TASKS:
                stacked = np.stack(pass_probs[t], axis=0)  # (MC_PASSES, batch, n_classes)
                accum_probs[t].append(stacked)

    for t in TASKS:
        all_stacked = np.concatenate(accum_probs[t], axis=1)  # (MC_PASSES, n_test, n_classes)
        mean_probs = all_stacked.mean(axis=0)
        std_probs = all_stacked.std(axis=0)
        for i, pid in enumerate(pid_order):
            row = test_df[test_df["PatientID"] == pid].iloc[0]
            if not bool(row[f"{t}_valid"]):
                continue
            phase7_oof_probs[t][pid] = mean_probs[i]
            phase7_oof_std[t][pid] = std_probs[i]
            phase7_oof_true[t][pid] = int(row[f"{t}_label"])

print("\nPhase 7 MC-Dropout inference complete.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 300})

def expected_calibration_error(confidences, correct, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_stats = []
    n = len(confidences)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            bin_stats.append((lo, hi, np.nan, np.nan, 0))
            continue
        bin_conf = confidences[mask].mean()
        bin_acc = correct[mask].mean()
        bin_stats.append((lo, hi, bin_conf, bin_acc, mask.sum()))
        ece += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return ece, bin_stats


phase7_summary = []
print(f"\n{'='*70}\nPhase 7: Point-estimate sanity check (mean of MC passes)\n{'='*70}")
for t in TASKS:
    pids = [p for p in master_df["PatientID"] if p in phase7_oof_true[t]]
    y_true = np.array([phase7_oof_true[t][p] for p in pids])
    probs = np.stack([phase7_oof_probs[t][p] for p in pids])
    preds = probs.argmax(axis=1)
    acc = (preds == y_true).mean()
    print(f"  {t.upper():8s} n={len(pids):4d}  Acc={acc:.3f}")
    phase7_summary.append({"task": t, "n": len(pids), "accuracy": acc})

# --- Calibration + confidence-stratified accuracy, using IDH (binary, primary endpoint) ---
task = "idh"
pids = [p for p in master_df["PatientID"] if p in phase7_oof_true[task]]
y_true = np.array([phase7_oof_true[task][p] for p in pids])
probs = np.stack([phase7_oof_probs[task][p] for p in pids])
preds = probs.argmax(axis=1)
correct = (preds == y_true).astype(float)
confidence = probs.max(axis=1)  # predicted-class confidence

ece, bin_stats = expected_calibration_error(confidence, correct, n_bins=10)
print(f"\n{'='*70}\nIDH Calibration -- Expected Calibration Error (ECE) = {ece:.4f}\n{'='*70}")
for lo, hi, bc, ba, n in bin_stats:
    if n > 0:
        print(f"  [{lo:.1f}-{hi:.1f}]  n={n:4d}  mean_confidence={bc:.3f}  accuracy={ba:.3f}")

# Reliability diagram
fig, ax = plt.subplots(figsize=(4.5, 4.5))
bin_confs = [b[2] for b in bin_stats if b[4] > 0]
bin_accs = [b[3] for b in bin_stats if b[4] > 0]
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.plot(bin_confs, bin_accs, marker="o", color="#1f6f8b", label=f"Model (ECE={ece:.3f})")
ax.set_xlabel("Mean predicted confidence")
ax.set_ylabel("Observed accuracy")
ax.set_title("IDH Reliability Diagram (MC-Dropout)", fontsize=9.5)
ax.legend(fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig("/kaggle/working/idh_reliability_diagram.png", dpi=300, bbox_inches="tight")
fig.savefig("/kaggle/working/idh_reliability_diagram.pdf", bbox_inches="tight")
plt.show()

# --- Confidence-stratified accuracy: does higher confidence = higher accuracy? ---
order = np.argsort(-confidence)  # most confident first
n = len(order)
deciles = np.array_split(order, 10)
print(f"\n{'='*70}\nIDH Accuracy by Confidence Decile (1=most confident, 10=least)\n{'='*70}")
decile_rows = []
for i, idx in enumerate(deciles):
    dec_acc = correct[idx].mean()
    dec_conf = confidence[idx].mean()
    print(f"  Decile {i+1:2d}  n={len(idx):3d}  mean_confidence={dec_conf:.3f}  accuracy={dec_acc:.3f}")
    decile_rows.append({"decile": i + 1, "n": len(idx), "mean_confidence": dec_conf, "accuracy": dec_acc})

top50 = order[:n // 2]
bottom50 = order[n // 2:]
print(f"\nTop 50% most confident  -- n={len(top50):3d}  accuracy={correct[top50].mean():.3f}")
print(f"Bottom 50% least confident -- n={len(bottom50):3d}  accuracy={correct[bottom50].mean():.3f}")
print(f"Overall accuracy -- n={n:3d}  accuracy={correct.mean():.3f}")

pd.DataFrame(decile_rows).to_csv("/kaggle/working/phase7_confidence_deciles.csv", index=False)
pd.DataFrame(phase7_summary).to_csv("/kaggle/working/phase7_summary.csv", index=False)
print("\nSaved: idh_reliability_diagram.png/.pdf, phase7_confidence_deciles.csv, phase7_summary.csv")

## Phase 8 — Explainability

- The attention weights from Phase 4 MIL pooling are already an explanation, no separate saliency method needed
- Quantitative evaluation: spatial overlap between high-attention slices and the tumour segmentation mask
- Qualitative evaluation: visualize attention-weighted slices for correctly versus incorrectly classified patients

In [ ]:
import base64, hashlib

EXPECTED_SHA256 = "4f8aa9e1e789bfd16a05752bb2d18a87c77c81d111c945b5210200ea5feed578"
EXPECTED_SIZE = 10883

_B64 = (
    "aW1wb3J0IG9zCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCBuaWJhYmVsIGFzIG5pYgppbXBvcnQgdG9yY2gKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApmcm9tIHNjaXB5Lm5kaW1hZ2UgaW1wb3J0IHpvb20KZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgcGVhcnNvbnIKZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgU3RyYXRpZmllZEtGb2xkLCB0cmFpbl90ZXN0X3NwbGl0CgpwcmludCgiPSIgKiA3MCkKcHJpbnQoIlBIQVNFIDg6IEV4cGxhaW5hYmlsaXR5IC0tIE1JTCBBdHRlbnRpb24gQW5hbHlzaXMiKQpwcmludCgiPSIgKiA3MCkKClNMSUNFX1NJWkUgPSAoMjI0LCAyMjQpCk5VTV9TTElDRVMgPSAxNQpNQVJHSU4gPSAyCgoKZGVmIG1ha2VfZGF0YXNldChkZik6CiAgICAjIEdsaW9tYUZlYXR1cmVEYXRhc2V0J3Mgc2lnbmF0dXJlIGNhbiB2YXJ5IGRlcGVuZGluZyBvbiB3aGljaCBwaGFzZQogICAgIyBsYXN0IHJlZGVmaW5lZCBpdCBpbiB0aGlzIHNlc3Npb24gKHNvbWUgdmVyc2lvbnMgdGFrZSBqdXN0IGBkZmAsCiAgICAjIG90aGVycyB0YWtlIGBkZiwgY2xpbmljYWxfY29scywgdGFza3NgKS4gSW5zcGVjdCBhbmQgY2FsbCBjb3JyZWN0bHkuCiAgICBpbXBvcnQgaW5zcGVjdAogICAgbl9wYXJhbXMgPSBsZW4oaW5zcGVjdC5zaWduYXR1cmUoR2xpb21hRmVhdHVyZURhdGFzZXQuX19pbml0X18pLnBhcmFtZXRlcnMpIC0gMSAgIyBleGNsdWRlIHNlbGYKICAgIGlmIG5fcGFyYW1zID49IDM6CiAgICAgICAgcmV0dXJuIEdsaW9tYUZlYXR1cmVEYXRhc2V0KGRmLCBDTElOSUNBTF9DT0xTLCBUQVNLUykKICAgIHJldHVybiBHbGlvbWFGZWF0dXJlRGF0YXNldChkZikKCgpkZWYgbWFza2VkX211bHRpdGFza19sb3NzX3NhZmUobG9naXRzLCBiYXRjaCwgdGFza3MsIGNsYXNzX3dlaWdodHMpOgogICAgdG90YWwgPSBOb25lCiAgICBmb3IgdCBpbiB0YXNrczoKICAgICAgICB2YWxpZCA9IGJhdGNoW2Yie3R9X3ZhbGlkIl0KICAgICAgICBpZiB2YWxpZC5zdW0oKSA9PSAwOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGxvc3MgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLmNyb3NzX2VudHJvcHkoCiAgICAg"
    "ICAgICAgIGxvZ2l0c1t0XVt2YWxpZF0sIGJhdGNoW2Yie3R9X2xhYmVsIl1bdmFsaWRdLCB3ZWlnaHQ9Y2xhc3Nfd2VpZ2h0cy5nZXQodCkKICAgICAgICApCiAgICAgICAgdG90YWwgPSBsb3NzIGlmIHRvdGFsIGlzIE5vbmUgZWxzZSB0b3RhbCArIGxvc3MKICAgIGlmIHRvdGFsIGlzIE5vbmU6CiAgICAgICAgYW55X3Rhc2sgPSBuZXh0KGl0ZXIobG9naXRzKSkKICAgICAgICB0b3RhbCA9IGxvZ2l0c1thbnlfdGFza10uc3VtKCkgKiAwLjAKICAgIHJldHVybiB0b3RhbAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU3RlcCAxOiBUcmFpbiB0aGUgbXVsdGktdGFzayArIGZ1c2lvbiBtb2RlbCB3aXRoIDUtZm9sZCBDVgojIChzYW1lIGFyY2hpdGVjdHVyZSwgc2FtZSBmb2xkIHNlZWQgYXMgU3RlcCA1KSBhbmQgY29sbGVjdAojIG91dC1vZi1mb2xkIGF0dGVudGlvbiB3ZWlnaHRzIHBlciBwYXRpZW50LgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpza2YgPSBTdHJhdGlmaWVkS0ZvbGQobl9zcGxpdHM9Tl9GT0xEUywgc2h1ZmZsZT1UcnVlLCByYW5kb21fc3RhdGU9NDIpCgpwaGFzZThfb29mX2F0dG4gPSB7fQpwaGFzZThfb29mX3ByZWQgPSB7fQpwaGFzZThfb29mX3Byb2IgPSB7fQpwaGFzZThfb29mX3RydWUgPSB7fQoKZm9yIGZvbGRfaWR4LCAodHJhaW5fdmFsX2lkeCwgdGVzdF9pZHgpIGluIGVudW1lcmF0ZShza2Yuc3BsaXQobWFzdGVyX2RmLCBtYXN0ZXJfZGZbImlkaF9sYWJlbCJdKSk6CiAgICBwcmludChmIlxuRm9sZCB7Zm9sZF9pZHggKyAxfS97Tl9GT0xEU30iKQogICAgdHJhaW5fdmFsX2RmID0gbWFzdGVyX2RmLmlsb2NbdHJhaW5fdmFsX2lkeF0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgdGVzdF9kZiA9IG1hc3Rlcl9kZi5pbG9jW3Rlc3RfaWR4XS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICB0cmFpbl9kZiwgdmFsX2RmID0gdHJhaW5fdGVzdF9zcGxpdCgKICAgICAgICB0cmFpbl92YWxfZGYsIHRlc3Rfc2l6ZT0wLjE1LCBzdHJhdGlmeT10cmFpbl92YWxfZGZbImlkaF9sYWJlbCJdLCByYW5kb21fc3Rh"
    "dGU9Zm9sZF9pZHgKICAgICkKCiAgICBjbGFzc193ZWlnaHRzID0gY29tcHV0ZV9jbGFzc193ZWlnaHRzKHRyYWluX2RmLCBUQVNLUykKICAgIHRyYWluX2xvYWRlciA9IERhdGFMb2FkZXIobWFrZV9kYXRhc2V0KHRyYWluX2RmKSwgYmF0Y2hfc2l6ZT1CQVRDSF9TSVpFLCBzaHVmZmxlPVRydWUpCiAgICB2YWxfbG9hZGVyID0gRGF0YUxvYWRlcihtYWtlX2RhdGFzZXQodmFsX2RmKSwgYmF0Y2hfc2l6ZT1CQVRDSF9TSVpFLCBzaHVmZmxlPUZhbHNlKQogICAgdGVzdF9sb2FkZXIgPSBEYXRhTG9hZGVyKG1ha2VfZGF0YXNldCh0ZXN0X2RmKSwgYmF0Y2hfc2l6ZT1CQVRDSF9TSVpFLCBzaHVmZmxlPUZhbHNlKQoKICAgIG1vZGVsID0gTXVsdGlUYXNrTUlMTW9kZWwoRU1CRURfRElNLCBDTElOSUNBTF9ESU0sIEFUVE5fSElEREVOLCBUQVNLUywgRFJPUE9VVF9QKS50byhkZXZpY2UpCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9TFIsIHdlaWdodF9kZWNheT1XRUlHSFRfREVDQVkpCgogICAgYmVzdF92YWxfbG9zcyA9IGZsb2F0KCJpbmYiKQogICAgYmVzdF9zdGF0ZSA9IE5vbmUKICAgIG5vX2ltcHJvdmUgPSAwCgogICAgZm9yIGVwb2NoIGluIHJhbmdlKE5VTV9FUE9DSFMpOgogICAgICAgIG1vZGVsLnRyYWluKCkKICAgICAgICBmb3IgYmF0Y2ggaW4gdHJhaW5fbG9hZGVyOgogICAgICAgICAgICBlbWIgPSBiYXRjaFsiZW1iZWRkaW5ncyJdLnRvKGRldmljZSkKICAgICAgICAgICAgY2xpbiA9IGJhdGNoWyJjbGluaWNhbCJdLnRvKGRldmljZSkKICAgICAgICAgICAgYiA9IHtrOiB2LnRvKGRldmljZSkgaWYgdG9yY2guaXNfdGVuc29yKHYpIGVsc2UgdiBmb3IgaywgdiBpbiBiYXRjaC5pdGVtcygpfQogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICAgICAgbG9naXRzLCBfID0gbW9kZWwoZW1iLCBjbGluKQogICAgICAgICAgICBsb3NzID0gbWFza2VkX211bHRpdGFza19sb3NzX3NhZmUobG9naXRzLCBiLCBUQVNLUywgY2xhc3Nfd2VpZ2h0cykKICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkKCiAgICAgICAgbW9kZWwuZXZhbCgpCiAgICAg"
    "ICAgdmFsX2xvc3MgPSAwLjAKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgZm9yIGJhdGNoIGluIHZhbF9sb2FkZXI6CiAgICAgICAgICAgICAgICBlbWIgPSBiYXRjaFsiZW1iZWRkaW5ncyJdLnRvKGRldmljZSkKICAgICAgICAgICAgICAgIGNsaW4gPSBiYXRjaFsiY2xpbmljYWwiXS50byhkZXZpY2UpCiAgICAgICAgICAgICAgICBiID0ge2s6IHYudG8oZGV2aWNlKSBpZiB0b3JjaC5pc190ZW5zb3IodikgZWxzZSB2IGZvciBrLCB2IGluIGJhdGNoLml0ZW1zKCl9CiAgICAgICAgICAgICAgICBsb2dpdHMsIF8gPSBtb2RlbChlbWIsIGNsaW4pCiAgICAgICAgICAgICAgICB2YWxfbG9zcyArPSBtYXNrZWRfbXVsdGl0YXNrX2xvc3Nfc2FmZShsb2dpdHMsIGIsIFRBU0tTLCBjbGFzc193ZWlnaHRzKS5pdGVtKCkKICAgICAgICB2YWxfbG9zcyAvPSBtYXgobGVuKHZhbF9sb2FkZXIpLCAxKQoKICAgICAgICBpZiB2YWxfbG9zcyA8IGJlc3RfdmFsX2xvc3MgLSAxZS00OgogICAgICAgICAgICBiZXN0X3ZhbF9sb3NzID0gdmFsX2xvc3MKICAgICAgICAgICAgYmVzdF9zdGF0ZSA9IHtrOiB2LmNsb25lKCkgZm9yIGssIHYgaW4gbW9kZWwuc3RhdGVfZGljdCgpLml0ZW1zKCl9CiAgICAgICAgICAgIG5vX2ltcHJvdmUgPSAwCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbm9faW1wcm92ZSArPSAxCiAgICAgICAgaWYgbm9faW1wcm92ZSA+PSBQQVRJRU5DRToKICAgICAgICAgICAgYnJlYWsKCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoYmVzdF9zdGF0ZSkKICAgIG1vZGVsLmV2YWwoKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIGJhdGNoIGluIHRlc3RfbG9hZGVyOgogICAgICAgICAgICBlbWIgPSBiYXRjaFsiZW1iZWRkaW5ncyJdLnRvKGRldmljZSkKICAgICAgICAgICAgY2xpbiA9IGJhdGNoWyJjbGluaWNhbCJdLnRvKGRldmljZSkKICAgICAgICAgICAgbG9naXRzLCBhdHRuID0gbW9kZWwoZW1iLCBjbGluKQogICAgICAgICAgICBwcm9icyA9IHRvcmNoLnNvZnRtYXgobG9naXRzWyJpZGgiXSwgZGltPTEpWzosIDFdLmNwdSgpLm51bXB5KCkKICAgICAgICAgICAgcHJlZHMgPSBsb2dpdHNbImlkaCJdLmFyZ21heChk"
    "aW09MSkuY3B1KCkubnVtcHkoKQogICAgICAgICAgICBhdHRuX25wID0gYXR0bi5jcHUoKS5udW1weSgpCiAgICAgICAgICAgIHRydWVfbnAgPSBiYXRjaFsiaWRoX2xhYmVsIl0ubnVtcHkoKQogICAgICAgICAgICBwaWRzID0gYmF0Y2hbIlBhdGllbnRJRCJdCiAgICAgICAgICAgIGZvciBpLCBwaWQgaW4gZW51bWVyYXRlKHBpZHMpOgogICAgICAgICAgICAgICAgcGhhc2U4X29vZl9hdHRuW3BpZF0gPSBhdHRuX25wW2ldCiAgICAgICAgICAgICAgICBwaGFzZThfb29mX3ByZWRbcGlkXSA9IGludChwcmVkc1tpXSkKICAgICAgICAgICAgICAgIHBoYXNlOF9vb2ZfcHJvYltwaWRdID0gZmxvYXQocHJvYnNbaV0pCiAgICAgICAgICAgICAgICBwaGFzZThfb29mX3RydWVbcGlkXSA9IGludCh0cnVlX25wW2ldKQoKICAgIHByaW50KGYiICBGb2xkIHtmb2xkX2lkeCArIDF9IHRlc3QgcGF0aWVudHM6IHtsZW4odGVzdF9kZil9LCAiCiAgICAgICAgICBmInRvdGFsIGNvbGxlY3RlZCBzbyBmYXI6IHtsZW4ocGhhc2U4X29vZl9hdHRuKX0iKQoKcHJpbnQoZiJcbkNvbGxlY3RlZCBPT0YgYXR0ZW50aW9uIHdlaWdodHMgZm9yIHtsZW4ocGhhc2U4X29vZl9hdHRuKX0gLyB7bGVuKG1hc3Rlcl9kZil9IHBhdGllbnRzIikKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFN0ZXAgMjogUXVhbnRpdGF0aXZlIG92ZXJsYXAgLS0gYXR0ZW50aW9uIHdlaWdodCB2cy4gdHVtb3VyCiMgY29udGVudCwgcmVjb21wdXRlZCBvbiB0aGUgc2FtZSB0dW1vdXItZ3VpZGVkIHotc2xpY2VzIHVzZWQKIyBpbiBQaGFzZSAyIHNsaWNlIGV4dHJhY3Rpb24uCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiB0dW1vdXJfel9yYW5nZShzZWcsIG1hcmdpbik6CiAgICBwcmVzZW50ID0gbnAuYW55KHNlZyA+IDAsIGF4aXM9KDAsIDEpKQogICAgaWR4ID0gbnAubm9uemVybyhwcmVzZW50KVswXQogICAgaWYgaWR4LnNpemUgPT0gMDoKICAgICAgICByZXR1cm4gTm9uZQogICAgel9taW4gPSBtYXgoMCwgaW50KGlkeC5taW4oKSkgLSBtYXJnaW4pCiAgICB6X21heCA9IG1p"
    "bihzZWcuc2hhcGVbMl0gLSAxLCBpbnQoaWR4Lm1heCgpKSArIG1hcmdpbikKICAgIHJldHVybiB6X21pbiwgel9tYXgKCgpkZWYgcmVzaXplMmQoaW1nLCBzaXplKToKICAgIGZhY3RvcnMgPSAoc2l6ZVswXSAvIGltZy5zaGFwZVswXSwgc2l6ZVsxXSAvIGltZy5zaGFwZVsxXSkKICAgIHJldHVybiB6b29tKGltZywgZmFjdG9ycywgb3JkZXI9MSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIHNsaWNlX3R1bW9yX2ZyYWN0aW9ucyhtYXNrX3BhdGgpOgogICAgc2VnID0gbnAuYXNhcnJheShuaWIubG9hZChtYXNrX3BhdGgpLmdldF9mZGF0YSgpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgel9yYW5nZSA9IHR1bW91cl96X3JhbmdlKHNlZywgTUFSR0lOKQogICAgaWYgel9yYW5nZSBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lLCBOb25lCiAgICB6X21pbiwgel9tYXggPSB6X3JhbmdlCiAgICBpZiB6X21heCA9PSB6X21pbjoKICAgICAgICB6X2luZGljZXMgPSBbel9taW5dICogTlVNX1NMSUNFUwogICAgZWxzZToKICAgICAgICB6X2luZGljZXMgPSBucC5yb3VuZChucC5saW5zcGFjZSh6X21pbiwgel9tYXgsIE5VTV9TTElDRVMpKS5hc3R5cGUoaW50KS50b2xpc3QoKQogICAgZnJhY3MgPSBbXQogICAgZm9yIHogaW4gel9pbmRpY2VzOgogICAgICAgIHNsID0gKHNlZ1s6LCA6LCB6XSA+IDApLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIHNsX3Jlc2l6ZWQgPSByZXNpemUyZChzbCwgU0xJQ0VfU0laRSkKICAgICAgICBmcmFjcy5hcHBlbmQoZmxvYXQoKHNsX3Jlc2l6ZWQgPiAwLjUpLm1lYW4oKSkpCiAgICByZXR1cm4gbnAuYXJyYXkoZnJhY3MpLCB6X2luZGljZXMKCgpwaWRfdG9fcm93ID0gbWFzdGVyX2RmLnNldF9pbmRleCgiUGF0aWVudElEIikKCmNvcnJlbGF0aW9ucyA9IFtdCnRvcDNfdnNfYWxsX3JhdGlvID0gW10KYXR0bl9tYXNzX29uX2hpZ2hfdHVtb3IgPSBbXQpwZXJfcGF0aWVudF9yZWNvcmRzID0gW10KCmZvciBwaWQsIGF0dG4gaW4gcGhhc2U4X29vZl9hdHRuLml0ZW1zKCk6CiAgICByb3cgPSBwaWRfdG9fcm93LmxvY1twaWRdCiAgICBmcmFjcywgel9pbmRpY2VzID0gc2xpY2VfdHVtb3JfZnJhY3Rpb25zKHJvd1siTWFza19QYXRoIl0pCiAgICBp"
    "ZiBmcmFjcyBpcyBOb25lIG9yIGZyYWNzLnN0ZCgpIDwgMWUtODoKICAgICAgICBjb250aW51ZQoKICAgIHIsIF8gPSBwZWFyc29ucihhdHRuLCBmcmFjcykKICAgIGNvcnJlbGF0aW9ucy5hcHBlbmQocikKCiAgICB0b3AzX2lkeCA9IG5wLmFyZ3NvcnQoYXR0bilbLTM6XQogICAgdG9wM19tZWFuX2ZyYWMgPSBmcmFjc1t0b3AzX2lkeF0ubWVhbigpCiAgICBhbGxfbWVhbl9mcmFjID0gZnJhY3MubWVhbigpCiAgICByYXRpbyA9IHRvcDNfbWVhbl9mcmFjIC8gKGFsbF9tZWFuX2ZyYWMgKyAxZS04KQogICAgdG9wM192c19hbGxfcmF0aW8uYXBwZW5kKHJhdGlvKQoKICAgIG1lZGlhbl9mcmFjID0gbnAubWVkaWFuKGZyYWNzKQogICAgaGlnaF90dW1vcl9tYXNrID0gZnJhY3MgPj0gbWVkaWFuX2ZyYWMKICAgIG1hc3MgPSBmbG9hdChhdHRuW2hpZ2hfdHVtb3JfbWFza10uc3VtKCkpCiAgICBhdHRuX21hc3Nfb25faGlnaF90dW1vci5hcHBlbmQobWFzcykKCiAgICBwZXJfcGF0aWVudF9yZWNvcmRzLmFwcGVuZCh7CiAgICAgICAgIlBhdGllbnRJRCI6IHBpZCwKICAgICAgICAiYXR0bl90dW1vcl9jb3JyIjogciwKICAgICAgICAidG9wM192c19hbGxfdHVtb3JfZnJhY19yYXRpbyI6IHJhdGlvLAogICAgICAgICJhdHRuX21hc3Nfb25faGlnaF90dW1vcl9zbGljZXMiOiBtYXNzLAogICAgICAgICJpZGhfdHJ1ZSI6IHBoYXNlOF9vb2ZfdHJ1ZVtwaWRdLAogICAgICAgICJpZGhfcHJlZCI6IHBoYXNlOF9vb2ZfcHJlZFtwaWRdLAogICAgICAgICJpZGhfcHJvYiI6IHBoYXNlOF9vb2ZfcHJvYltwaWRdLAogICAgfSkKCmNvcnJlbGF0aW9ucyA9IG5wLmFycmF5KGNvcnJlbGF0aW9ucykKdG9wM192c19hbGxfcmF0aW8gPSBucC5hcnJheSh0b3AzX3ZzX2FsbF9yYXRpbykKYXR0bl9tYXNzX29uX2hpZ2hfdHVtb3IgPSBucC5hcnJheShhdHRuX21hc3Nfb25faGlnaF90dW1vcikKCnByaW50KCJcbiIgKyAiPSIgKiA3MCkKcHJpbnQoIlBoYXNlIDg6IFF1YW50aXRhdGl2ZSBhdHRlbnRpb24tdHVtb3VyIG92ZXJsYXAiKQpwcmludCgiPSIgKiA3MCkKcHJpbnQoZiJQYXRpZW50cyBhbmFseXplZDoge2xlbihjb3JyZWxhdGlvbnMpfSIpCnByaW50KGYiTWVhbiBwZXItcGF0aWVudCBhdHRl"
    "bnRpb25+dHVtb3VyLWZyYWN0aW9uIGNvcnJlbGF0aW9uOiAiCiAgICAgIGYie2NvcnJlbGF0aW9ucy5tZWFuKCk6LjNmfSAobWVkaWFuIHtucC5tZWRpYW4oY29ycmVsYXRpb25zKTouM2Z9KSIpCnByaW50KGYiRnJhY3Rpb24gb2YgcGF0aWVudHMgd2l0aCBwb3NpdGl2ZSBjb3JyZWxhdGlvbjogeyhjb3JyZWxhdGlvbnMgPiAwKS5tZWFuKCk6LjNmfSIpCnByaW50KGYiTWVhbiB0dW1vdXItZnJhY3Rpb24gcmF0aW8sIHRvcC0zIGF0dGVuZGVkIHNsaWNlcyB2cy4gYWxsIDE1IHNsaWNlczogIgogICAgICBmInt0b3AzX3ZzX2FsbF9yYXRpby5tZWFuKCk6LjNmfXggICg+MS4wIG1lYW5zIGF0dGVudGlvbiBjb25jZW50cmF0ZXMgb24gbW9yZSB0dW1vdXItcmljaCBzbGljZXMpIikKcHJpbnQoZiJNZWFuIGF0dGVudGlvbiBtYXNzIHBsYWNlZCBvbiBhYm92ZS1tZWRpYW4tdHVtb3VyIHNsaWNlczogIgogICAgICBmInthdHRuX21hc3Nfb25faGlnaF90dW1vci5tZWFuKCk6LjNmfSAgKGNoYW5jZSBsZXZlbCA9IDAuNTAwKSIpCgpyZXN1bHRzX2RmID0gcGQuRGF0YUZyYW1lKHBlcl9wYXRpZW50X3JlY29yZHMpCnJlc3VsdHNfZGYudG9fY3N2KCIva2FnZ2xlL3dvcmtpbmcvcGhhc2U4X2F0dGVudGlvbl90dW1vcl9vdmVybGFwLmNzdiIsIGluZGV4PUZhbHNlKQpwcmludCgiXG5TYXZlZDogL2thZ2dsZS93b3JraW5nL3BoYXNlOF9hdHRlbnRpb25fdHVtb3Jfb3ZlcmxhcC5jc3YiKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU3RlcCAzOiBRdWFsaXRhdGl2ZSAtLSBhdHRlbnRpb24td2VpZ2h0ZWQgc2xpY2UgZ3JpZHMgZm9yCiMgY29ycmVjdGx5IGFuZCBpbmNvcnJlY3RseSBjbGFzc2lmaWVkIHBhdGllbnRzLgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpkZWYgcGxvdF9wYXRpZW50X2F0dGVudGlvbihwaWQsIHNhdmVfc3VmZml4KToKICAgIGJhZyA9IG5wLmxvYWQoZiIva2FnZ2xlL3dvcmtpbmcvc2xpY2VzL3twaWR9Lm5weiIpWyJzbGljZXMiXSAgIyAoMTUsIDQsIDIyNCwgMjI0KQogICAgYXR0biA9IHBoYXNlOF9vb2ZfYXR0bltw"
    "aWRdCiAgICByb3cgPSBwaWRfdG9fcm93LmxvY1twaWRdCiAgICBmcmFjcywgXyA9IHNsaWNlX3R1bW9yX2ZyYWN0aW9ucyhyb3dbIk1hc2tfUGF0aCJdKQoKICAgIG4gPSBiYWcuc2hhcGVbMF0KICAgIG5jb2xzID0gNQogICAgbnJvd3MgPSBpbnQobnAuY2VpbChuIC8gbmNvbHMpKQogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKG5yb3dzLCBuY29scywgZmlnc2l6ZT0obmNvbHMgKiAyLjIsIG5yb3dzICogMi40KSkKICAgIGF4ZXMgPSBucC5hcnJheShheGVzKS5yZXNoYXBlKC0xKQogICAgZm9yIGkgaW4gcmFuZ2Uobik6CiAgICAgICAgYXggPSBheGVzW2ldCiAgICAgICAgZmxhaXJfaW1nID0gYmFnW2ksIDNdICAjIEZMQUlSIGNoYW5uZWwKICAgICAgICBheC5pbXNob3coZmxhaXJfaW1nLCBjbWFwPSJncmF5IikKICAgICAgICBmcmFjX3R4dCA9IGYie2ZyYWNzW2ldOi4yZn0iIGlmIGZyYWNzIGlzIG5vdCBOb25lIGVsc2UgIm4vYSIKICAgICAgICBheC5zZXRfdGl0bGUoZiJ3PXthdHRuW2ldOi4yZn1cbnR1bW91cj17ZnJhY190eHR9IiwgZm9udHNpemU9OCkKICAgICAgICBheC5zZXRfeHRpY2tzKFtdKQogICAgICAgIGF4LnNldF95dGlja3MoW10pCiAgICBmb3IgaiBpbiByYW5nZShuLCBsZW4oYXhlcykpOgogICAgICAgIGF4ZXNbal0uYXhpcygib2ZmIikKCiAgICB0cnVlX2xibCA9IHBoYXNlOF9vb2ZfdHJ1ZVtwaWRdCiAgICBwcmVkX2xibCA9IHBoYXNlOF9vb2ZfcHJlZFtwaWRdCiAgICBwcm9iID0gcGhhc2U4X29vZl9wcm9iW3BpZF0KICAgIGNvcnJlY3QgPSAiQ09SUkVDVCIgaWYgdHJ1ZV9sYmwgPT0gcHJlZF9sYmwgZWxzZSAiSU5DT1JSRUNUIgogICAgZmlnLnN1cHRpdGxlKAogICAgICAgIGYiUGF0aWVudCB7cGlkfSAtLSBJREggdHJ1ZT17dHJ1ZV9sYmx9IHByZWQ9e3ByZWRfbGJsfSAocD17cHJvYjouMmZ9KSBbe2NvcnJlY3R9XSIsCiAgICAgICAgZm9udHNpemU9MTEsCiAgICApCiAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgIG91dF9wbmcgPSBmIi9rYWdnbGUvd29ya2luZy9waGFzZThfYXR0ZW50aW9uX3tzYXZlX3N1ZmZpeH1fe3BpZH0ucG5nIgogICAgZmlnLnNhdmVmaWcob3V0X3BuZywgZHBpPTIwMCwgYmJveF9pbmNoZXM9InRpZ2h0IikK"
    "ICAgIHBsdC5zaG93KCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICBwcmludChmIlNhdmVkOiB7b3V0X3BuZ30iKQoKCmNvcnJlY3RfZGYgPSByZXN1bHRzX2RmW3Jlc3VsdHNfZGZbImlkaF90cnVlIl0gPT0gcmVzdWx0c19kZlsiaWRoX3ByZWQiXV0uc29ydF92YWx1ZXMoCiAgICAiaWRoX3Byb2IiLCBhc2NlbmRpbmc9RmFsc2UKKQppbmNvcnJlY3RfZGYgPSByZXN1bHRzX2RmW3Jlc3VsdHNfZGZbImlkaF90cnVlIl0gIT0gcmVzdWx0c19kZlsiaWRoX3ByZWQiXV0KCmV4YW1wbGVfY29ycmVjdCA9IGNvcnJlY3RfZGYuaGVhZCgyKVsiUGF0aWVudElEIl0udG9saXN0KCkKZXhhbXBsZV9pbmNvcnJlY3QgPSBpbmNvcnJlY3RfZGYuaGVhZCgyKVsiUGF0aWVudElEIl0udG9saXN0KCkKCnByaW50KGYiXG5QbG90dGluZyBhdHRlbnRpb24gbWFwcyBmb3IgY29ycmVjdCBleGFtcGxlczoge2V4YW1wbGVfY29ycmVjdH0iKQpmb3IgcGlkIGluIGV4YW1wbGVfY29ycmVjdDoKICAgIHBsb3RfcGF0aWVudF9hdHRlbnRpb24ocGlkLCAiY29ycmVjdCIpCgpwcmludChmIlxuUGxvdHRpbmcgYXR0ZW50aW9uIG1hcHMgZm9yIGluY29ycmVjdCBleGFtcGxlczoge2V4YW1wbGVfaW5jb3JyZWN0fSIpCmZvciBwaWQgaW4gZXhhbXBsZV9pbmNvcnJlY3Q6CiAgICBwbG90X3BhdGllbnRfYXR0ZW50aW9uKHBpZCwgImluY29ycmVjdCIpCgpwcmludCgiXG4iICsgIj0iICogNzApCnByaW50KCJQaGFzZSA4IGNvbXBsZXRlLiIpCnByaW50KCI9IiAqIDcwKQo="
)

_data = base64.b64decode("".join(_B64))
_actual_sha256 = hashlib.sha256(_data).hexdigest()
assert len(_data) == EXPECTED_SIZE, f"Size mismatch: got {len(_data)}, expected {EXPECTED_SIZE}"
assert _actual_sha256 == EXPECTED_SHA256, f"Checksum mismatch: got {_actual_sha256}"

with open("/kaggle/working/phase8_explainability.py", "wb") as f:
    f.write(_data)

print(f"OK -- wrote /kaggle/working/phase8_explainability.py ({len(_data)} bytes, checksum verified)")

In [ ]:
   _phase8_globals = dict(globals())
   _phase8_globals["__name__"] = "phase8_module"
   exec(open("/kaggle/working/phase8_explainability.py").read(), _phase8_globals)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# pull the saved results back out (exec didn't dump these into top-level globals)
phase8_oof_attn = _phase8_globals["phase8_oof_attn"]
phase8_oof_true = _phase8_globals["phase8_oof_true"]
phase8_oof_pred = _phase8_globals["phase8_oof_pred"]
phase8_oof_prob = _phase8_globals["phase8_oof_prob"]
pid_to_row = _phase8_globals["pid_to_row"]
slice_tumor_fractions = _phase8_globals["slice_tumor_fractions"]
results_df = _phase8_globals["results_df"]


def plot_patient_attention_v2(pid, save_suffix):
    bag = np.load(f"/kaggle/working/slices/{pid}.npz")["slices"]
    attn = phase8_oof_attn[pid]
    row = pid_to_row.loc[pid]
    fracs, _ = slice_tumor_fractions(row["Mask_Path"])

    n = bag.shape[0]
    ncols = 5
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.6, nrows * 3.0))
    axes = np.array(axes).reshape(-1)
    for i in range(n):
        ax = axes[i]
        ax.imshow(bag[i, 3], cmap="gray")
        frac_txt = f"{fracs[i]:.2f}" if fracs is not None else "n/a"
        ax.set_xlabel(f"w={attn[i]:.3f}\ntumour={frac_txt}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    for j in range(n, len(axes)):
        axes[j].axis("off")

    true_lbl = phase8_oof_true[pid]
    pred_lbl = phase8_oof_pred[pid]
    prob = phase8_oof_prob[pid]
    correct = "CORRECT" if true_lbl == pred_lbl else "INCORRECT"
    fig.suptitle(f"Patient {pid} -- IDH true={true_lbl} pred={pred_lbl} (p={prob:.2f}) [{correct}]", fontsize=12)
    plt.subplots_adjust(hspace=0.6, wspace=0.3, top=0.90)
    out_png = f"/kaggle/working/phase8_attention_v2_{save_suffix}_{pid}.png"
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {out_png}")


correct_df = results_df[results_df["idh_true"] == results_df["idh_pred"]].sort_values("idh_prob", ascending=False)
incorrect_df = results_df[results_df["idh_true"] != results_df["idh_pred"]]

for pid in correct_df.head(2)["PatientID"].tolist():
    plot_patient_attention_v2(pid, "correct")
for pid in incorrect_df.head(2)["PatientID"].tolist():
    plot_patient_attention_v2(pid, "incorrect")

In [ ]:
import base64, hashlib

EXPECTED_SHA256 = "4154e40469928ddf5862cdbaa49fe11e4e2182a9eb1edafa9e8d1a9548bd4ea6"
EXPECTED_SIZE = 8122

_B64 = (
    "aW1wb3J0IG9zCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKaW1wb3J0IG1hdHBsb3RsaWIuaW1hZ2UgYXMgbXBpbWcKCnByaW50KCI9IiAqIDcwKQpwcmludCgiUmVzdWx0cyBDb25zb2xpZGF0aW9uIC0tIGNvbWJpbmVkIHN1bW1hcnkgZmlndXJlICsgbWFzdGVyIHRhYmxlIikKcHJpbnQoIj0iICogNzApCgpXT1JLID0gIi9rYWdnbGUvd29ya2luZyIKCgpkZWYgdHJ5X3JlYWRfY3N2KHBhdGgpOgogICAgaWYgb3MucGF0aC5leGlzdHMocGF0aCk6CiAgICAgICAgZGYgPSBwZC5yZWFkX2NzdihwYXRoKQogICAgICAgIHByaW50KGYiXG5Mb2FkZWQge3BhdGh9ICAoc2hhcGU9e2RmLnNoYXBlfSkiKQogICAgICAgIHByaW50KCJDb2x1bW5zOiIsIGRmLmNvbHVtbnMudG9saXN0KCkpCiAgICAgICAgcmV0dXJuIGRmCiAgICBwcmludChmIlxuTUlTU0lORzoge3BhdGh9IikKICAgIHJldHVybiBOb25lCgoKcGhhc2U1X2RmID0gdHJ5X3JlYWRfY3N2KGYie1dPUkt9L3BoYXNlNV9hYmxhdGlvbl9yZXN1bHRzLmNzdiIpCnBoYXNlNl9kZiA9IHRyeV9yZWFkX2NzdihmIntXT1JLfS9waGFzZTZfbGFiZWxfZWZmaWNpZW5jeV9yZXN1bHRzLmNzdiIpCnBoYXNlN19zdW1tYXJ5X2RmID0gdHJ5X3JlYWRfY3N2KGYie1dPUkt9L3BoYXNlN19zdW1tYXJ5LmNzdiIpCnBoYXNlN19kZWNpbGVfZGYgPSB0cnlfcmVhZF9jc3YoZiJ7V09SS30vcGhhc2U3X2NvbmZpZGVuY2VfZGVjaWxlcy5jc3YiKQpwaGFzZThfZGYgPSB0cnlfcmVhZF9jc3YoZiJ7V09SS30vcGhhc2U4X2F0dGVudGlvbl90dW1vcl9vdmVybGFwLmNzdiIpCgpmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMiwgMiwgZmlnc2l6ZT0oMTMsIDExKSkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgUGFuZWwgMTogUGhhc2UgNSBhYmxhdGlvbiAtLSBJREggQVVDIGJ5IGNvbmRpdGlvbgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpheCA9IGF4ZXNbMCwgMF0KdHJ5OgogICAgZGYgPSBwaGFzZTVfZGYuY29weSgpCiAgICBp"
    "ZGhfY29sID0gTm9uZQogICAgY29uZF9jb2wgPSBOb25lCiAgICBmb3IgYyBpbiBkZi5jb2x1bW5zOgogICAgICAgIGxjID0gYy5sb3dlcigpCiAgICAgICAgaWYgImNvbmRpdGlvbiIgaW4gbGMgb3IgbGMgPT0gInRhc2siIGFuZCBGYWxzZToKICAgICAgICAgICAgcGFzcwogICAgY29uZF9jYW5kaWRhdGVzID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiAiY29uZGl0aW9uIiBpbiBjLmxvd2VyKCkgb3IgYy5sb3dlcigpID09ICJtb2RlbCJdCiAgICBjb25kX2NvbCA9IGNvbmRfY2FuZGlkYXRlc1swXSBpZiBjb25kX2NhbmRpZGF0ZXMgZWxzZSBkZi5jb2x1bW5zWzBdCgogICAgaWYgInRhc2siIGluIFtjLmxvd2VyKCkgZm9yIGMgaW4gZGYuY29sdW1uc106CiAgICAgICAgdGFza19jb2wgPSBbYyBmb3IgYyBpbiBkZi5jb2x1bW5zIGlmIGMubG93ZXIoKSA9PSAidGFzayJdWzBdCiAgICAgICAgYXVjX2NvbCA9IFtjIGZvciBjIGluIGRmLmNvbHVtbnMgaWYgImF1YyIgaW4gYy5sb3dlcigpXVswXQogICAgICAgIHN1YiA9IGRmW2RmW3Rhc2tfY29sXS5hc3R5cGUoc3RyKS5zdHIubG93ZXIoKSA9PSAiaWRoIl0KICAgICAgICBsYWJlbHMgPSBzdWJbY29uZF9jb2xdLmFzdHlwZShzdHIpLnRvbGlzdCgpCiAgICAgICAgdmFsdWVzID0gc3ViW2F1Y19jb2xdLmFzdHlwZShmbG9hdCkudG9saXN0KCkKICAgIGVsc2U6CiAgICAgICAgYXVjX2NvbHMgPSBbYyBmb3IgYyBpbiBkZi5jb2x1bW5zIGlmICJpZGgiIGluIGMubG93ZXIoKSBhbmQgImF1YyIgaW4gYy5sb3dlcigpXQogICAgICAgIGF1Y19jb2wgPSBhdWNfY29sc1swXSBpZiBhdWNfY29scyBlbHNlIFtjIGZvciBjIGluIGRmLmNvbHVtbnMgaWYgImF1YyIgaW4gYy5sb3dlcigpXVswXQogICAgICAgIGxhYmVscyA9IGRmW2NvbmRfY29sXS5hc3R5cGUoc3RyKS50b2xpc3QoKQogICAgICAgIHZhbHVlcyA9IGRmW2F1Y19jb2xdLmFzdHlwZShmbG9hdCkudG9saXN0KCkKCiAgICBiYXJzID0gYXguYmFyKHJhbmdlKGxlbihsYWJlbHMpKSwgdmFsdWVzLCBjb2xvcj0iIzRDNzJCMCIpCiAgICBheC5zZXRfeHRpY2tzKHJhbmdlKGxlbihsYWJlbHMpKSkKICAgIGF4LnNldF94dGlja2xhYmVscyhsYWJlbHMsIHJvdGF0aW9uPTI1LCBo"
    "YT0icmlnaHQiLCBmb250c2l6ZT04KQogICAgYXguc2V0X3lsYWJlbCgiSURIIEFVQyIpCiAgICBheC5zZXRfdGl0bGUoIlBoYXNlIDUgLS0gQWJsYXRpb246IElESCBBVUMgYnkgY29uZGl0aW9uIikKICAgIGF4LnNldF95bGltKDAsIDEuMCkKICAgIGZvciBpLCB2IGluIGVudW1lcmF0ZSh2YWx1ZXMpOgogICAgICAgIGF4LnRleHQoaSwgdiArIDAuMDEsIGYie3Y6LjNmfSIsIGhhPSJjZW50ZXIiLCBmb250c2l6ZT04KQpleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICBheC50ZXh0KDAuNSwgMC41LCBmIkNvdWxkIG5vdCBhdXRvLXBsb3QgUGhhc2UgNS5cbkNoZWNrIGNvbHVtbnMgYWJvdmUuXG4oe2V9KSIsCiAgICAgICAgICAgIGhhPSJjZW50ZXIiLCB2YT0iY2VudGVyIiwgd3JhcD1UcnVlLCBmb250c2l6ZT05KQogICAgYXguc2V0X3RpdGxlKCJQaGFzZSA1IC0tIEFibGF0aW9uIChzZWUgY29uc29sZSBmb3IgY29sdW1ucykiKQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBQYW5lbCAyOiBQaGFzZSA2IGxhYmVsLWVmZmljaWVuY3kgY3VydmUKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KYXggPSBheGVzWzAsIDFdCnRyeToKICAgIGltZ19wYXRoID0gZiJ7V09SS30vbGFiZWxfZWZmaWNpZW5jeV9jdXJ2ZS5wbmciCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhpbWdfcGF0aCk6CiAgICAgICAgaW1nID0gbXBpbWcuaW1yZWFkKGltZ19wYXRoKQogICAgICAgIGF4Lmltc2hvdyhpbWcpCiAgICAgICAgYXguYXhpcygib2ZmIikKICAgICAgICBheC5zZXRfdGl0bGUoIlBoYXNlIDYgLS0gTGFiZWwtRWZmaWNpZW5jeSBDdXJ2ZSIpCiAgICBlbHNlOgogICAgICAgIGRmID0gcGhhc2U2X2RmLmNvcHkoKQogICAgICAgIGZyYWNfY29sID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiAiZnJhYyIgaW4gYy5sb3dlcigpXVswXQogICAgICAgIGF1Y19jb2xzID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiAiaWRoIiBpbiBjLmxvd2VyKCkgYW5kICJhdWMiIGluIGMubG93ZXIoKV0KICAgICAgICBhdWNfY29sID0gYXVjX2NvbHNbMF0gaWYgYXVjX2Nv"
    "bHMgZWxzZSBbYyBmb3IgYyBpbiBkZi5jb2x1bW5zIGlmICJhdWMiIGluIGMubG93ZXIoKV1bMF0KICAgICAgICBzdWIgPSBkZi5zb3J0X3ZhbHVlcyhmcmFjX2NvbCkKICAgICAgICBheC5wbG90KHN1YltmcmFjX2NvbF0sIHN1YlthdWNfY29sXSwgbWFya2VyPSJvIiwgY29sb3I9IiNERDg0NTIiKQogICAgICAgIGF4LnNldF94bGFiZWwoIkZyYWN0aW9uIG9mIGxhYmVsbGVkIHRyYWluaW5nIGRhdGEiKQogICAgICAgIGF4LnNldF95bGFiZWwoIklESCBBVUMiKQogICAgICAgIGF4LnNldF90aXRsZSgiUGhhc2UgNiAtLSBMYWJlbC1FZmZpY2llbmN5IEN1cnZlIikKICAgICAgICBheC5zZXRfeWxpbSgwLCAxLjApCmV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgIGF4LnRleHQoMC41LCAwLjUsIGYiQ291bGQgbm90IGxvYWQgUGhhc2UgNiBwYW5lbC5cbih7ZX0pIiwgaGE9ImNlbnRlciIsIHZhPSJjZW50ZXIiLCB3cmFwPVRydWUsIGZvbnRzaXplPTkpCiAgICBheC5zZXRfdGl0bGUoIlBoYXNlIDYgLS0gTGFiZWwtRWZmaWNpZW5jeSAoc2VlIGNvbnNvbGUpIikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgUGFuZWwgMzogUGhhc2UgNyByZWxpYWJpbGl0eSBkaWFncmFtIChyZXVzZSBzYXZlZCBpbWFnZSkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KYXggPSBheGVzWzEsIDBdCnRyeToKICAgIGltZ19wYXRoID0gZiJ7V09SS30vaWRoX3JlbGlhYmlsaXR5X2RpYWdyYW0ucG5nIgogICAgaW1nID0gbXBpbWcuaW1yZWFkKGltZ19wYXRoKQogICAgYXguaW1zaG93KGltZykKICAgIGF4LmF4aXMoIm9mZiIpCiAgICBheC5zZXRfdGl0bGUoIlBoYXNlIDcgLS0gQ2FsaWJyYXRpb24gKE1DLURyb3BvdXQpIikKZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgYXgudGV4dCgwLjUsIDAuNSwgZiJDb3VsZCBub3QgbG9hZCByZWxpYWJpbGl0eSBkaWFncmFtLlxuKHtlfSkiLCBoYT0iY2VudGVyIiwgdmE9ImNlbnRlciIsIHdyYXA9VHJ1ZSwgZm9udHNpemU9OSkKICAgIGF4LnNldF90aXRsZSgiUGhhc2UgNyAtLSBDYWxpYnJhdGlvbiAoc2VlIGNvbnNv"
    "bGUpIikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgUGFuZWwgNDogUGhhc2UgOCAtLSBhdHRlbnRpb24gdnMgdHVtb3VyLWZyYWN0aW9uIHN1bW1hcnkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KYXggPSBheGVzWzEsIDFdCnRyeToKICAgIGRmID0gcGhhc2U4X2RmLmNvcHkoKQogICAgY29ycl9jb2wgPSBbYyBmb3IgYyBpbiBkZi5jb2x1bW5zIGlmICJjb3JyIiBpbiBjLmxvd2VyKCldWzBdCiAgICBheC5oaXN0KGRmW2NvcnJfY29sXS5kcm9wbmEoKSwgYmlucz0zMCwgY29sb3I9IiM1NUE4NjgiLCBlZGdlY29sb3I9IndoaXRlIikKICAgIG1lYW5fY29yciA9IGRmW2NvcnJfY29sXS5tZWFuKCkKICAgIGF4LmF4dmxpbmUobWVhbl9jb3JyLCBjb2xvcj0iYmxhY2siLCBsaW5lc3R5bGU9Ii0tIiwgbGluZXdpZHRoPTEpCiAgICBheC5heHZsaW5lKDAsIGNvbG9yPSJyZWQiLCBsaW5lc3R5bGU9IjoiLCBsaW5ld2lkdGg9MSkKICAgIGF4LnNldF94bGFiZWwoIlBlci1wYXRpZW50IGF0dGVudGlvbn50dW1vdXItZnJhY3Rpb24gY29ycmVsYXRpb24iKQogICAgYXguc2V0X3lsYWJlbCgiTnVtYmVyIG9mIHBhdGllbnRzIikKICAgIGF4LnNldF90aXRsZShmIlBoYXNlIDggLS0gQXR0ZW50aW9uL1R1bW91ciBPdmVybGFwXG4obWVhbiByID0ge21lYW5fY29ycjouM2Z9LCBuPXtsZW4oZGYpfSkiKQpleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICBheC50ZXh0KDAuNSwgMC41LCBmIkNvdWxkIG5vdCBwbG90IFBoYXNlIDggcGFuZWwuXG4oe2V9KSIsIGhhPSJjZW50ZXIiLCB2YT0iY2VudGVyIiwgd3JhcD1UcnVlLCBmb250c2l6ZT05KQogICAgYXguc2V0X3RpdGxlKCJQaGFzZSA4IC0tIEV4cGxhaW5hYmlsaXR5IChzZWUgY29uc29sZSkiKQoKZmlnLnN1cHRpdGxlKCJSZXN1bHRzIE92ZXJ2aWV3IC0tIFBoYXNlcyA1LTgiLCBmb250c2l6ZT0xNSwgeT0wLjk5NSkKZmlnLnRpZ2h0X2xheW91dChyZWN0PVswLCAwLCAxLCAwLjk3XSkKZmlnLnNhdmVmaWcoZiJ7V09SS30vcmVzdWx0c19vdmVydmlld19maWd1cmUucG5nIiwgZHBpPTIwMCwgYmJv"
    "eF9pbmNoZXM9InRpZ2h0IikKZmlnLnNhdmVmaWcoZiJ7V09SS30vcmVzdWx0c19vdmVydmlld19maWd1cmUucGRmIiwgYmJveF9pbmNoZXM9InRpZ2h0IikKcGx0LnNob3coKQpwcmludChmIlxuU2F2ZWQ6IHtXT1JLfS9yZXN1bHRzX292ZXJ2aWV3X2ZpZ3VyZS5wbmciKQpwcmludChmIlNhdmVkOiB7V09SS30vcmVzdWx0c19vdmVydmlld19maWd1cmUucGRmIikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgTWFzdGVyIHJlc3VsdHMgdGFibGUgKGJlc3QtZWZmb3J0LCByb2J1c3QgdG8gY29sdW1uLW5hbWUgZHJpZnQpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCnJvd3MgPSBbXQp0cnk6CiAgICByb3dzLmFwcGVuZCh7IlBoYXNlIjogIjUgLSBBYmxhdGlvbiIsICJNZXRyaWMiOiAiU2VlIHBoYXNlNV9hYmxhdGlvbl9yZXN1bHRzLmNzdiBmb3IgZnVsbCB0YWJsZSIsCiAgICAgICAgICAgICAgICAgIlZhbHVlIjogIiJ9KQpleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwoKaWYgcGhhc2U2X2RmIGlzIG5vdCBOb25lOgogICAgdHJ5OgogICAgICAgIGZyYWNfY29sID0gW2MgZm9yIGMgaW4gcGhhc2U2X2RmLmNvbHVtbnMgaWYgImZyYWMiIGluIGMubG93ZXIoKV1bMF0KICAgICAgICBhdWNfY29scyA9IFtjIGZvciBjIGluIHBoYXNlNl9kZi5jb2x1bW5zIGlmICJpZGgiIGluIGMubG93ZXIoKSBhbmQgImF1YyIgaW4gYy5sb3dlcigpXQogICAgICAgIGF1Y19jb2wgPSBhdWNfY29sc1swXSBpZiBhdWNfY29scyBlbHNlIFtjIGZvciBjIGluIHBoYXNlNl9kZi5jb2x1bW5zIGlmICJhdWMiIGluIGMubG93ZXIoKV1bMF0KICAgICAgICBmb3IgXywgciBpbiBwaGFzZTZfZGYuc29ydF92YWx1ZXMoZnJhY19jb2wpLml0ZXJyb3dzKCk6CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsiUGhhc2UiOiAiNiAtIExhYmVsIGVmZmljaWVuY3kiLAogICAgICAgICAgICAgICAgICAgICAgICAgICJNZXRyaWMiOiBmIklESCBBVUMgQCB7cltmcmFjX2NvbF0qMTAwOi4wZn0lIGxhYmVscyIgaWYgcltmcmFjX2NvbF0gPD0gMQogICAgICAgICAgICAgICAgICAg"
    "ICAgICAgIGVsc2UgZiJJREggQVVDIEAge3JbZnJhY19jb2xdOi4wZn0lIGxhYmVscyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIlZhbHVlIjogcm91bmQoZmxvYXQoclthdWNfY29sXSksIDQpfSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIkNvdWxkIG5vdCBzdW1tYXJpemUgUGhhc2UgNiBpbnRvIG1hc3RlciB0YWJsZToge2V9IikKCmlmIHBoYXNlN19zdW1tYXJ5X2RmIGlzIG5vdCBOb25lOgogICAgdHJ5OgogICAgICAgIGZvciBfLCByIGluIHBoYXNlN19zdW1tYXJ5X2RmLml0ZXJyb3dzKCk6CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsiUGhhc2UiOiAiNyAtIENhbGlicmF0aW9uIiwgIk1ldHJpYyI6IHN0cihyLnRvX2RpY3QoKSksICJWYWx1ZSI6ICIifSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIkNvdWxkIG5vdCBzdW1tYXJpemUgUGhhc2UgNyBpbnRvIG1hc3RlciB0YWJsZToge2V9IikKCmlmIHBoYXNlOF9kZiBpcyBub3QgTm9uZToKICAgIHRyeToKICAgICAgICBjb3JyX2NvbCA9IFtjIGZvciBjIGluIHBoYXNlOF9kZi5jb2x1bW5zIGlmICJjb3JyIiBpbiBjLmxvd2VyKCldWzBdCiAgICAgICAgcm93cy5hcHBlbmQoeyJQaGFzZSI6ICI4IC0gRXhwbGFpbmFiaWxpdHkiLCAiTWV0cmljIjogIk1lYW4gYXR0ZW50aW9ufnR1bW91ciBjb3JyZWxhdGlvbiIsCiAgICAgICAgICAgICAgICAgICAgICAiVmFsdWUiOiByb3VuZChmbG9hdChwaGFzZThfZGZbY29ycl9jb2xdLm1lYW4oKSksIDQpfSkKICAgICAgICByb3dzLmFwcGVuZCh7IlBoYXNlIjogIjggLSBFeHBsYWluYWJpbGl0eSIsICJNZXRyaWMiOiAiRnJhY3Rpb24gb2YgcGF0aWVudHMgd2l0aCBwb3NpdGl2ZSBjb3JyZWxhdGlvbiIsCiAgICAgICAgICAgICAgICAgICAgICAiVmFsdWUiOiByb3VuZChmbG9hdCgocGhhc2U4X2RmW2NvcnJfY29sXSA+IDApLm1lYW4oKSksIDQpfSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIkNvdWxkIG5vdCBzdW1tYXJpemUgUGhhc2UgOCBpbnRvIG1hc3RlciB0YWJsZToge2V9IikKCm1hc3Rlcl9kZiA9IHBkLkRhdGFGcmFtZShyb3dzKQptYXN0ZXJfZGYudG9fY3N2"
    "KGYie1dPUkt9L21hc3Rlcl9yZXN1bHRzX3RhYmxlLmNzdiIsIGluZGV4PUZhbHNlKQpwcmludChmIlxuU2F2ZWQ6IHtXT1JLfS9tYXN0ZXJfcmVzdWx0c190YWJsZS5jc3YiKQpwcmludCgiXG5NYXN0ZXIgdGFibGUgcHJldmlldzoiKQpwcmludChtYXN0ZXJfZGYudG9fc3RyaW5nKGluZGV4PUZhbHNlKSkKCnByaW50KCJcbiIgKyAiPSIgKiA3MCkKcHJpbnQoIkNvbnNvbGlkYXRpb24gY29tcGxldGUuIikKcHJpbnQoIj0iICogNzApCg=="
)

_data = base64.b64decode("".join(_B64))
_actual_sha256 = hashlib.sha256(_data).hexdigest()
assert len(_data) == EXPECTED_SIZE, f"Size mismatch: got {len(_data)}, expected {EXPECTED_SIZE}"
assert _actual_sha256 == EXPECTED_SHA256, f"Checksum mismatch: got {_actual_sha256}"

with open("/kaggle/working/phase9_summary.py", "wb") as f:
    f.write(_data)

print(f"OK -- wrote /kaggle/working/phase9_summary.py ({len(_data)} bytes, checksum verified)")

In [ ]:
exec(open("/kaggle/working/phase9_summary.py").read())

In [ ]:
"""
======================================================================
PHASE 5 (PATCHED): Baseline and Ablation Comparisons -- REAL PIPELINE
======================================================================
Phase 5 script with three targeted patches applied:

  PATCH A -- New condition (v): "Radiomics + clinical only" -- a plain
             classifier trained on the 209-dim clinical/radiomic vector
             ALONE (no image embeddings, no attention pooling), under
             the identical cached 5-fold split. This lets the paper
             report how much of the fusion gain is coming from the
             image branch vs. the tabular features alone.

  PATCH B -- Per-patient out-of-fold (OOF) predictions are now written
             to disk for EVERY condition/task, not just the pooled
             summary metrics. Saved to phase5_oof_predictions.csv with
             columns: PatientID, condition, task, num_classes,
             true_label, pred_label, prob_0, prob_1, prob_2 (if 3-class).
             This is what a companion DeLong/permutation significance
             test needs (see phase5_significance_test.py).

  PATCH C -- Fixed feature-scaling leakage: z-score normalization of
             the four base clinical scalars (Age/Volume/T1CE/FLAIR) and
             all radiomics columns is now computed PER TRAINING FOLD
             (fit on train_df only, applied to val_df/test_df) instead
             of globally over the whole cohort before fold-splitting.
             This only affects conditions that actually use
             clinical_cols: (iv) fusion and the new (v) structured-only
             condition. Conditions (i)-(iii) are untouched because they
             never see clinical_cols.

Everything else -- file paths, hyperparameters, model architectures,
the 5-fold split cache, bootstrap CI, metric definitions -- is
UNCHANGED from the original Phase 5 script, so conditions (i)-(iv) should
reproduce numbers close to the earlier run (condition (iv)'s AUC
may shift slightly because of Patch C's normalization fix removing a
small amount of leakage).

Usage: run as-is.
It reuses the existing cached artifacts (features/, radiomics,
clinical cache, fold-split cache) -- delete phase5_fold_splits.json
only if you want a fresh split.
======================================================================
"""

import os
import json
import time
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
from scipy.ndimage import zoom
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix, average_precision_score,
)
from sklearn.utils.class_weight import compute_class_weight

# ----------------------------------------------------------------------
# CONFIG -- unchanged from the Step-5 run
# ----------------------------------------------------------------------
FEATURES_DIR = "/kaggle/working/features"
SLICES_DIR = "/kaggle/working/slices"
METADATA_PATH = "/kaggle/input/datasets/tano1974/glioma/UTSW_Glioma_Metadata-2-1.tsv"
MASTER_PATH = "/kaggle/working/master_dataset_fets_with_slices.csv"
RADIOMICS_PATH = "/kaggle/working/radiomics_features.csv"
CACHED_CLINICAL_PATH = "/kaggle/working/clinical_features_cache.csv"
SPLIT_CACHE = "/kaggle/working/phase5_fold_splits.json"
RESULTS_CSV = "/kaggle/working/phase5_ablation_results.csv"
OOF_CSV = "/kaggle/working/phase5_oof_predictions.csv"          # NEW (Patch B)

EMBED_DIM = 1280
ATTN_HIDDEN = 128
DROPOUT_P = 0.4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 40
PATIENCE = 5
BATCH_SIZE = 16
LR = 1e-3
N_FOLDS = 5
N_BOOTSTRAP = 300
SEED = 42

# Baseline 3D CNN -- smaller/faster since it trains end-to-end from raw volumes
BASELINE_EPOCHS = 25
BASELINE_PATIENCE = 5
BASELINE_LR = 1e-4
BASELINE_BATCH = 2
VOLUME_SHAPE = (64, 128, 128)  # (D, H, W) after resampling

# Structured-only (radiomics+clinical) classifier -- Patch A
STRUCT_EPOCHS = 60
STRUCT_PATIENCE = 8
STRUCT_LR = 1e-3
STRUCT_HIDDEN = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: GPU not detected -- training will be much slower. "
          "Check Settings -> Accelerator in Kaggle.")

GRADE_MAP = {2.0: 0, 3.0: 1, 4.0: 2}
BINARY_MAP = {
    "MGMT": {"unmethylated": 0, "methylated": 1},
    "1p19Q CODEL": {"non co-deleted": 0, "co-deleted": 1},
}
TASKS = {
    "idh": {"num_classes": 2}, "grade": {"num_classes": 3},
    "mgmt": {"num_classes": 2}, "codel": {"num_classes": 2},
}

# Global accumulator for per-patient OOF predictions across ALL conditions
# (Patch B) -- flushed to OOF_CSV at the end of main().
ALL_OOF_ROWS = []


# ======================================================================
# STEP 2+3: Clinical + radiomics merge, label encoding
# ----------------------------------------------------------------------
# PATCH C: this now returns the RAW (un-normalized) clinical/radiomic
# columns plus the list of column names that need z-score normalization.
# Normalization itself is done per-training-fold inside the condition
# runners below, not here.
# ======================================================================
def load_nii(path):
    return np.asarray(nib.load(path).get_fdata(), dtype=np.float32)


def build_master_df():
    master_df = pd.read_csv(MASTER_PATH)
    master_df = master_df[master_df["has_slices"] == True].reset_index(drop=True)

    if os.path.exists(CACHED_CLINICAL_PATH):
        cache_df = pd.read_csv(CACHED_CLINICAL_PATH)
        master_df = master_df.merge(cache_df, on="PatientID", how="left")
    else:
        raise FileNotFoundError(
            f"{CACHED_CLINICAL_PATH} not found -- run Phase 4's Step 2 cell first "
            f"(this file should already exist from your prior run)."
        )

    master_df["SexEncoded"] = master_df["Sex"].map({"M": 0, "F": 1})

    # PATCH C: keep these RAW (no global normalization). The output
    # column names are unchanged (AgeNorm etc.) purely so the rest of
    # the pipeline (column lists, cached artifacts) doesn't need to
    # change -- but the values here are the raw quantities, not
    # z-scored yet. Per-fold normalization happens later.
    base_clinical_cols = ["AgeNorm", "SexEncoded", "VolumeNorm", "T1CENorm", "FLAIRNorm"]
    for col, out_col in [("Age", "AgeNorm"), ("TumourVolumeVoxels", "VolumeNorm"),
                          ("T1CEMeanIntensity", "T1CENorm"), ("FLAIRMeanIntensity", "FLAIRNorm")]:
        master_df[out_col] = master_df[col].astype(np.float32)

    radiomics_cols = []
    if os.path.exists(RADIOMICS_PATH):
        radiomics_df = pd.read_csv(RADIOMICS_PATH)
        master_df = master_df.merge(radiomics_df, on="PatientID", how="left")
        radiomics_cols = [c for c in radiomics_df.columns if c != "PatientID"]
        # NOTE (Patch C): no normalization and no fillna(0.0) here anymore --
        # both are now done per-fold (fit on train, applied to val/test) in
        # normalize_clinical_per_fold() below, so NaNs are filled with the
        # TRAINING fold's mean rather than a fixed 0.0 computed globally.
    else:
        print("No radiomics file -- using lightweight clinical features only")

    # Columns that actually need z-score normalization (excludes the
    # already-binary SexEncoded column).
    cols_to_normalize = ["AgeNorm", "VolumeNorm", "T1CENorm", "FLAIRNorm"] + radiomics_cols
    clinical_cols = base_clinical_cols + radiomics_cols
    print(f"Total clinical/radiomic feature dimension: {len(clinical_cols)}")

    raw_meta = pd.read_csv(METADATA_PATH, sep="\t")
    raw_meta = raw_meta.rename(columns={"Subject ID": "PatientID"})
    if "MGMT" not in master_df.columns:
        master_df = master_df.merge(raw_meta[["PatientID", "MGMT", "1p19Q CODEL"]],
                                     on="PatientID", how="left")

    def encode_labels(row):
        out = {}
        out["idh_label"] = int(row["IDH_Label"])
        out["idh_valid"] = True
        grade_val = row.get("Grade", np.nan)
        if grade_val in GRADE_MAP:
            out["grade_label"] = GRADE_MAP[grade_val]
            out["grade_valid"] = True
        else:
            out["grade_label"] = 0
            out["grade_valid"] = False
        mgmt_val = row.get("MGMT", None)
        if isinstance(mgmt_val, str) and mgmt_val.strip() in BINARY_MAP["MGMT"]:
            out["mgmt_label"] = BINARY_MAP["MGMT"][mgmt_val.strip()]
            out["mgmt_valid"] = True
        else:
            out["mgmt_label"] = 0
            out["mgmt_valid"] = False
        codel_val = row.get("1p19Q CODEL", None)
        is_idh_mutant = int(row["IDH_Label"]) == 1
        if is_idh_mutant and isinstance(codel_val, str) and codel_val.strip() in BINARY_MAP["1p19Q CODEL"]:
            out["codel_label"] = BINARY_MAP["1p19Q CODEL"][codel_val.strip()]
            out["codel_valid"] = True
        else:
            out["codel_label"] = 0
            out["codel_valid"] = False
        return pd.Series(out)

    labels_df = master_df.apply(encode_labels, axis=1)
    master_df = pd.concat([master_df, labels_df], axis=1)

    print("\nTask label availability:")
    for task in TASKS:
        print(f"  {task}: {master_df[f'{task}_valid'].sum()} / {len(master_df)} valid")

    return master_df, clinical_cols, cols_to_normalize


# ======================================================================
# PATCH C: per-training-fold normalization helper
# ======================================================================
def normalize_clinical_per_fold(train_df, val_df, test_df, cols_to_normalize):
    """Fits mean/std (and NaN-fill value) on train_df ONLY, applies the
    same statistics to val_df/test_df. Returns new copies -- does not
    mutate the inputs (so the same master_df can be reused across folds
    and conditions).
    """
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()
    for col in cols_to_normalize:
        train_mean = train_df[col].mean()
        train_std = train_df[col].std()
        fill_value = train_mean if not np.isnan(train_mean) else 0.0
        for d in (train_df, val_df, test_df):
            d[col] = d[col].fillna(fill_value)
            d[col] = (d[col] - train_mean) / (train_std + 1e-8)
    return train_df, val_df, test_df


# ======================================================================
# Fixed 5-fold split, stratified on IDH -- shared across ALL conditions
# ======================================================================
def get_or_create_folds(master_df):
    if os.path.exists(SPLIT_CACHE):
        print(f"Loading cached fold splits from {SPLIT_CACHE}")
        with open(SPLIT_CACHE, "r") as f:
            folds = json.load(f)
        return folds

    print("No cached splits found -- creating fixed 5-fold split (stratified on IDH)...")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds = []
    for train_val_idx, test_idx in skf.split(master_df, master_df["idh_label"]):
        folds.append({
            "train_val": master_df.iloc[train_val_idx]["PatientID"].tolist(),
            "test": master_df.iloc[test_idx]["PatientID"].tolist(),
        })
    with open(SPLIT_CACHE, "w") as f:
        json.dump(folds, f)
    print(f"Cached fold splits to {SPLIT_CACHE} -- all conditions will reuse these")
    return folds


# ======================================================================
# Shared model pieces (unchanged)
# ======================================================================
class AttentionMIL(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout_p):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.Tanh(),
                                        nn.Dropout(dropout_p), nn.Linear(hidden_dim, 1))

    def forward(self, x):
        w = torch.softmax(self.attention(x), dim=1)
        return (x * w).sum(dim=1), w.squeeze(-1)


class MILModel(nn.Module):
    """Generalized version of your MultiTaskMILModel -- clinical_dim=0 gives
    the image-only variant used by conditions (ii) and (iii)."""
    def __init__(self, embed_dim, tasks, attn_hidden=ATTN_HIDDEN, dropout_p=DROPOUT_P, clinical_dim=0):
        super().__init__()
        self.pooling = AttentionMIL(embed_dim, attn_hidden, dropout_p)
        self.use_fusion = clinical_dim > 0
        self.fusion_dropout = nn.Dropout(dropout_p)
        fused_dim = embed_dim + clinical_dim
        self.heads = nn.ModuleDict({
            t: nn.Sequential(nn.Linear(fused_dim, 64), nn.ReLU(),
                              nn.Dropout(dropout_p), nn.Linear(64, c["num_classes"]))
            for t, c in tasks.items()
        })

    def forward(self, x, clinical=None):
        pooled, attn = self.pooling(x)
        if self.use_fusion:
            fused = self.fusion_dropout(torch.cat([pooled, clinical], dim=1))
        else:
            fused = self.fusion_dropout(pooled)
        return {t: h(fused) for t, h in self.heads.items()}, attn


class GliomaFeatureDataset(Dataset):
    """Reuses your cached embeddings -- no backbone forward pass needed."""
    def __init__(self, df, clinical_cols, tasks):
        self.df = df.reset_index(drop=True)
        self.clinical_cols = clinical_cols
        self.tasks = tasks

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["PatientID"]
        embeddings = np.load(os.path.join(FEATURES_DIR, f"{pid}.npy"))
        clinical = row[self.clinical_cols].values.astype(np.float32)
        sample = {"embeddings": torch.from_numpy(embeddings).float(),
                  "clinical": torch.from_numpy(clinical).float(), "PatientID": pid}
        for task in self.tasks:
            sample[f"{task}_label"] = torch.tensor(row[f"{task}_label"], dtype=torch.long)
            sample[f"{task}_valid"] = torch.tensor(bool(row[f"{task}_valid"]), dtype=torch.bool)
        return sample


# ------------------------------------------------------------------
# PATCH A: dataset for the structured-only (no-image) condition
# ------------------------------------------------------------------
class ClinicalOnlyDataset(Dataset):
    """Same label/validity plumbing as GliomaFeatureDataset, but no
    image embeddings at all -- just the clinical/radiomic vector."""
    def __init__(self, df, clinical_cols, tasks):
        self.df = df.reset_index(drop=True)
        self.clinical_cols = clinical_cols
        self.tasks = tasks

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["PatientID"]
        clinical = row[self.clinical_cols].values.astype(np.float32)
        sample = {"clinical": torch.from_numpy(clinical).float(), "PatientID": pid}
        for task in self.tasks:
            sample[f"{task}_label"] = torch.tensor(row[f"{task}_label"], dtype=torch.long)
            sample[f"{task}_valid"] = torch.tensor(bool(row[f"{task}_valid"]), dtype=torch.bool)
        return sample


class StructuredOnlyModel(nn.Module):
    """Plain MLP over the clinical/radiomic vector, one head per task --
    architecturally matched to MILModel's heads (Linear->ReLU->Dropout->
    Linear) so any AUC gap vs. condition (iv) is attributable to the
    image branch, not to a bigger/smaller classifier."""
    def __init__(self, clinical_dim, tasks, hidden=STRUCT_HIDDEN, dropout_p=DROPOUT_P):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(clinical_dim, hidden), nn.ReLU(), nn.Dropout(dropout_p))
        self.heads = nn.ModuleDict({
            t: nn.Linear(hidden, c["num_classes"]) for t, c in tasks.items()
        })

    def forward(self, clinical):
        h = self.trunk(clinical)
        return {t: head(h) for t, head in self.heads.items()}


def masked_multitask_loss(logits, batch, tasks, class_weights):
    # BUGFIX: the original version initialized `total = 0.0` (a plain Python
    # float) and only ever reassigned it to a tensor if at least one task had
    # a valid label in this batch. For single-task conditions on sparsely
    # labeled tasks (mgmt: 277/619 valid, codel: 140/619 valid), a shuffled
    # batch of size BATCH_SIZE can easily contain ZERO patients with a valid
    # label for that one task -- so `total` stayed a float, and the caller's
    # `loss.backward()` / `.item()` then crashed with
    # "AttributeError: 'float' object has no attribute 'backward'".
    # Fix: always return a tensor. When no task contributes a real loss,
    # return a zero-valued tensor still attached to the graph (via the
    # model's own output), so backward()/step()/.item() are harmless no-ops
    # for that batch instead of crashing.
    total = None
    for t in tasks:
        valid = batch[f"{t}_valid"]
        if valid.sum() == 0:
            continue
        loss = nn.functional.cross_entropy(logits[t][valid], batch[f"{t}_label"][valid],
                                            weight=class_weights.get(t))
        total = loss if total is None else total + loss
    if total is None:
        any_logits = next(iter(logits.values()))
        total = any_logits.sum() * 0.0
    return total


def compute_class_weights(train_df, tasks):
    cw = {}
    for task, cfg in tasks.items():
        valid_mask = train_df[f"{task}_valid"]
        if valid_mask.sum() < 2:
            continue
        labels = train_df.loc[valid_mask, f"{task}_label"].values
        present = np.unique(labels)
        if len(present) < 2:
            continue
        w = compute_class_weight("balanced", classes=present, y=labels)
        full_w = np.ones(cfg["num_classes"], dtype=np.float32)
        for cls, ww in zip(present, w):
            full_w[int(cls)] = ww
        cw[task] = torch.tensor(full_w, dtype=torch.float32).to(device)
    return cw


def bootstrap_auc_ci(y_true, probs, num_classes, n_boot=N_BOOTSTRAP, ci=95, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        pp = probs[idx]
        try:
            if num_classes == 2:
                boot_aucs.append(roc_auc_score(yt, pp[:, 1]))
            else:
                boot_aucs.append(roc_auc_score(yt, pp, multi_class="ovr"))
        except ValueError:
            continue
    if len(boot_aucs) == 0:
        return float("nan"), float("nan"), float("nan")
    return (np.mean(boot_aucs), np.percentile(boot_aucs, (100 - ci) / 2),
            np.percentile(boot_aucs, 100 - (100 - ci) / 2))


def summarize_task(task, num_classes, oof_labels, oof_logits, oof_valid, all_pids):
    pids = [p for p in all_pids if oof_valid[task].get(p, False)]
    if len(pids) < 2:
        return None
    yv = np.array([oof_labels[p][task] for p in pids])
    probs = np.stack([torch.softmax(torch.tensor(oof_logits[task][p]), dim=0).numpy() for p in pids])
    preds = probs.argmax(axis=1)
    if len(np.unique(yv)) < 2:
        return None

    acc = accuracy_score(yv, preds)
    bal_acc = balanced_accuracy_score(yv, preds)
    f1 = f1_score(yv, preds, average="macro")
    precision = precision_score(yv, preds, average="macro", zero_division=0)
    recall = recall_score(yv, preds, average="macro", zero_division=0)
    cm = confusion_matrix(yv, preds, labels=list(range(num_classes)))

    try:
        if num_classes == 2:
            auc = roc_auc_score(yv, probs[:, 1])
            pr_auc = average_precision_score(yv, probs[:, 1])
        else:
            auc = roc_auc_score(yv, probs, multi_class="ovr")
            pr_auc = float("nan")
    except ValueError:
        auc, pr_auc = float("nan"), float("nan")

    mean_auc, ci_lo, ci_hi = bootstrap_auc_ci(yv, probs, num_classes)

    return {
        "task": task, "n": len(pids), "accuracy": acc, "balanced_accuracy": bal_acc,
        "macro_f1": f1, "macro_precision": precision, "macro_recall": recall,
        "auc": auc, "pr_auc": pr_auc, "auc_ci_low": ci_lo, "auc_ci_high": ci_hi,
        "confusion_matrix": cm.tolist(),
    }


# ------------------------------------------------------------------
# PATCH B: turn a condition's raw OOF dicts into per-patient CSV rows
# ------------------------------------------------------------------
def record_oof_predictions(condition_name, task, num_classes, oof_labels, oof_logits, oof_valid, all_pids):
    pids = [p for p in all_pids if oof_valid[task].get(p, False)]
    for pid in pids:
        probs = torch.softmax(torch.tensor(oof_logits[task][pid]), dim=0).numpy()
        row = {
            "PatientID": pid,
            "condition": condition_name,
            "task": task,
            "num_classes": num_classes,
            "true_label": int(oof_labels[pid][task]),
            "pred_label": int(np.argmax(probs)),
        }
        for c in range(num_classes):
            row[f"prob_{c}"] = float(probs[c])
        ALL_OOF_ROWS.append(row)


# ======================================================================
# Conditions (ii), (iii), (iv) -- MIL variants, all reuse cached embeddings
# ======================================================================
def run_mil_condition(master_df, clinical_cols, cols_to_normalize, folds, tasks, use_fusion, condition_name):
    print(f"\n{'=' * 70}\nCONDITION: {condition_name}\n{'=' * 70}")
    oof_logits = {t: {} for t in tasks}
    oof_labels = {}
    oof_valid = {t: {} for t in tasks}
    clinical_dim = len(clinical_cols) if use_fusion else 0

    for fold_idx, fold in enumerate(folds):
        print(f"\n--- Fold {fold_idx + 1}/{N_FOLDS} ---")
        train_val_df = master_df[master_df["PatientID"].isin(fold["train_val"])].reset_index(drop=True)
        test_df = master_df[master_df["PatientID"].isin(fold["test"])].reset_index(drop=True)
        train_df, val_df = train_test_split(train_val_df, test_size=0.15,
                                             stratify=train_val_df["idh_label"], random_state=fold_idx)

        # PATCH C: fit clinical normalization on train_df only, apply to val/test.
        # Only matters when use_fusion=True (otherwise clinical_cols aren't consumed),
        # but it's harmless/cheap to always do it.
        if use_fusion:
            train_df, val_df, test_df = normalize_clinical_per_fold(
                train_df, val_df, test_df, cols_to_normalize)

        class_weights = compute_class_weights(train_df, tasks)
        train_loader = DataLoader(GliomaFeatureDataset(train_df, clinical_cols, tasks),
                                   batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(GliomaFeatureDataset(val_df, clinical_cols, tasks),
                                 batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(GliomaFeatureDataset(test_df, clinical_cols, tasks),
                                  batch_size=BATCH_SIZE, shuffle=False)

        model = MILModel(EMBED_DIM, tasks, clinical_dim=clinical_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val_loss = float("inf")
        best_state = None
        no_improve = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for batch in train_loader:
                emb = batch["embeddings"].to(device)
                clin = batch["clinical"].to(device) if use_fusion else None
                b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                optimizer.zero_grad()
                logits, _ = model(emb, clin)
                loss = masked_multitask_loss(logits, b, tasks, class_weights)
                loss.backward()
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    emb = batch["embeddings"].to(device)
                    clin = batch["clinical"].to(device) if use_fusion else None
                    b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                    logits, _ = model(emb, clin)
                    val_loss += masked_multitask_loss(logits, b, tasks, class_weights).item()
            val_loss /= len(val_loader)

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                break

        model.load_state_dict(best_state)
        print(f"Fold {fold_idx + 1} best val_loss: {best_val_loss:.4f} (model restored)")

        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                emb = batch["embeddings"].to(device)
                clin = batch["clinical"].to(device) if use_fusion else None
                logits, _ = model(emb, clin)
                pids = batch["PatientID"]
                for i, pid in enumerate(pids):
                    oof_labels[pid] = {t: batch[f"{t}_label"][i].item() for t in tasks}
                    for t in tasks:
                        oof_logits[t][pid] = logits[t][i].cpu().numpy()
                        oof_valid[t][pid] = batch[f"{t}_valid"][i].item()

    results = []
    all_pids = master_df["PatientID"].tolist()
    for t in tasks:
        r = summarize_task(t, tasks[t]["num_classes"], oof_labels, oof_logits, oof_valid, all_pids)
        # Patch B: always record predictions, even if summarize_task can't
        # compute a summary (e.g. too few valid patients) -- the raw
        # predictions are still useful downstream.
        record_oof_predictions(condition_name, t, tasks[t]["num_classes"],
                                oof_labels, oof_logits, oof_valid, all_pids)
        if r is None:
            continue
        r["condition"] = condition_name
        results.append(r)
        print(f"  {t.upper():8s} n={r['n']:4d}  AUC={r['auc']:.3f}  "
              f"Acc={r['accuracy']:.3f}  BalAcc={r['balanced_accuracy']:.3f}")
    return results


# ======================================================================
# PATCH A: Condition (v) -- Radiomics + clinical only (no images)
# ======================================================================
def run_structured_only_condition(master_df, clinical_cols, cols_to_normalize, folds, tasks,
                                   condition_name="(v) Radiomics+clinical only"):
    print(f"\n{'=' * 70}\nCONDITION: {condition_name}\n{'=' * 70}")
    oof_logits = {t: {} for t in tasks}
    oof_labels = {}
    oof_valid = {t: {} for t in tasks}
    clinical_dim = len(clinical_cols)

    for fold_idx, fold in enumerate(folds):
        print(f"\n--- Fold {fold_idx + 1}/{N_FOLDS} ---")
        train_val_df = master_df[master_df["PatientID"].isin(fold["train_val"])].reset_index(drop=True)
        test_df = master_df[master_df["PatientID"].isin(fold["test"])].reset_index(drop=True)
        train_df, val_df = train_test_split(train_val_df, test_size=0.15,
                                             stratify=train_val_df["idh_label"], random_state=fold_idx)

        # Patch C: per-fold normalization, same as the fusion condition.
        train_df, val_df, test_df = normalize_clinical_per_fold(
            train_df, val_df, test_df, cols_to_normalize)

        class_weights = compute_class_weights(train_df, tasks)
        train_loader = DataLoader(ClinicalOnlyDataset(train_df, clinical_cols, tasks),
                                   batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(ClinicalOnlyDataset(val_df, clinical_cols, tasks),
                                 batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(ClinicalOnlyDataset(test_df, clinical_cols, tasks),
                                  batch_size=BATCH_SIZE, shuffle=False)

        model = StructuredOnlyModel(clinical_dim, tasks).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=STRUCT_LR, weight_decay=WEIGHT_DECAY)

        best_val_loss = float("inf")
        best_state = None
        no_improve = 0

        for epoch in range(STRUCT_EPOCHS):
            model.train()
            for batch in train_loader:
                clin = batch["clinical"].to(device)
                b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                optimizer.zero_grad()
                logits = model(clin)
                loss = masked_multitask_loss(logits, b, tasks, class_weights)
                loss.backward()
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    clin = batch["clinical"].to(device)
                    b = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                    logits = model(clin)
                    val_loss += masked_multitask_loss(logits, b, tasks, class_weights).item()
            val_loss /= len(val_loader)

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= STRUCT_PATIENCE:
                break

        model.load_state_dict(best_state)
        print(f"Fold {fold_idx + 1} best val_loss: {best_val_loss:.4f} (model restored)")

        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                clin = batch["clinical"].to(device)
                logits = model(clin)
                pids = batch["PatientID"]
                for i, pid in enumerate(pids):
                    oof_labels[pid] = {t: batch[f"{t}_label"][i].item() for t in tasks}
                    for t in tasks:
                        oof_logits[t][pid] = logits[t][i].cpu().numpy()
                        oof_valid[t][pid] = batch[f"{t}_valid"][i].item()

    results = []
    all_pids = master_df["PatientID"].tolist()
    for t in tasks:
        r = summarize_task(t, tasks[t]["num_classes"], oof_labels, oof_logits, oof_valid, all_pids)
        record_oof_predictions(condition_name, t, tasks[t]["num_classes"],
                                oof_labels, oof_logits, oof_valid, all_pids)
        if r is None:
            continue
        r["condition"] = condition_name
        results.append(r)
        print(f"  {t.upper():8s} n={r['n']:4d}  AUC={r['auc']:.3f}  "
              f"Acc={r['accuracy']:.3f}  BalAcc={r['balanced_accuracy']:.3f}")
    return results


# ======================================================================
# Condition (i) -- Baseline 3D CNN, fully-supervised, IDH only (unchanged)
# ======================================================================
def zscore(vol, mask):
    voxels = vol[mask]
    if voxels.size == 0:
        return vol
    mean, std = voxels.mean(), voxels.std()
    return np.clip((vol - mean) / (std + 1e-6), -5, 5)


def load_and_resample_volume(row, target_shape=VOLUME_SHAPE):
    t1 = load_nii(row["T1_Path"])
    t1ce = load_nii(row["T1CE_Path"])
    t2 = load_nii(row["T2_Path"])
    flair = load_nii(row["FLAIR_Path"])
    brain_mask = (t1 > 0) | (t1ce > 0) | (t2 > 0) | (flair > 0)

    channels = []
    for vol in [t1, t1ce, t2, flair]:
        vol = zscore(vol, brain_mask)
        vol = np.transpose(vol, (2, 0, 1))  # (H,W,D) -> (D,H,W)
        factors = [t / s for t, s in zip(target_shape, vol.shape)]
        channels.append(zoom(vol, factors, order=1).astype(np.float32))
    return np.stack(channels, axis=0)  # (4, D, H, W)


class VolumeDataset(Dataset):
    def __init__(self, df, cache_dir="/kaggle/working/baseline_volumes"):
        self.df = df.reset_index(drop=True)
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["PatientID"]
        cache_path = os.path.join(self.cache_dir, f"{pid}.npy")
        if os.path.exists(cache_path):
            vol = np.load(cache_path)
        else:
            vol = load_and_resample_volume(row)
            np.save(cache_path, vol)
        return {"volume": torch.from_numpy(vol).float(),
                "label": torch.tensor(int(row["idh_label"]), dtype=torch.long),
                "PatientID": pid}


class Baseline3DCNN(nn.Module):
    def __init__(self, n_classes=2, in_channels=4):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv3d(cin, cout, 3, padding=1), nn.BatchNorm3d(cout), nn.ReLU(),
                nn.MaxPool3d(2)
            )
        self.features = nn.Sequential(block(in_channels, 32), block(32, 64),
                                       block(64, 128), block(128, 256))
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(256, 128), nn.ReLU(),
                                   nn.Dropout(0.3), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.gap(self.features(x)))


def run_baseline_condition(master_df, folds, condition_name="(i) Baseline 3D CNN (fully-supervised, IDH)"):
    print(f"\n{'=' * 70}\nCONDITION: {condition_name}\n{'=' * 70}")
    oof_logits, oof_labels = {}, {}

    for fold_idx, fold in enumerate(folds):
        print(f"\n--- Fold {fold_idx + 1}/{N_FOLDS} ---")
        train_val_df = master_df[master_df["PatientID"].isin(fold["train_val"])].reset_index(drop=True)
        test_df = master_df[master_df["PatientID"].isin(fold["test"])].reset_index(drop=True)
        train_df, val_df = train_test_split(train_val_df, test_size=0.15,
                                             stratify=train_val_df["idh_label"], random_state=fold_idx)

        train_loader = DataLoader(VolumeDataset(train_df), batch_size=BASELINE_BATCH, shuffle=True)
        val_loader = DataLoader(VolumeDataset(val_df), batch_size=BASELINE_BATCH, shuffle=False)
        test_loader = DataLoader(VolumeDataset(test_df), batch_size=BASELINE_BATCH, shuffle=False)

        model = Baseline3DCNN(n_classes=2).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=BASELINE_LR, weight_decay=WEIGHT_DECAY)

        best_val_loss = float("inf")
        best_state = None
        no_improve = 0
        fold_start = time.time()

        for epoch in range(BASELINE_EPOCHS):
            epoch_start = time.time()
            model.train()
            n_batches = 0
            for batch in train_loader:
                vol = batch["volume"].to(device)
                label = batch["label"].to(device)
                optimizer.zero_grad()
                loss = nn.functional.cross_entropy(model(vol), label)
                loss.backward()
                optimizer.step()
                n_batches += 1

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    vol = batch["volume"].to(device)
                    label = batch["label"].to(device)
                    val_loss += nn.functional.cross_entropy(model(vol), label).item()
            val_loss /= max(len(val_loader), 1)

            epoch_time = time.time() - epoch_start
            sec_per_batch = epoch_time / max(n_batches, 1)
            elapsed = time.time() - fold_start
            remaining_epochs_this_fold = BASELINE_EPOCHS - (epoch + 1)
            eta_this_fold = remaining_epochs_this_fold * epoch_time
            folds_remaining = N_FOLDS - (fold_idx + 1)
            eta_total = eta_this_fold + folds_remaining * BASELINE_EPOCHS * epoch_time
            print(f"    epoch {epoch + 1}/{BASELINE_EPOCHS}  "
                  f"val_loss={val_loss:.4f}  "
                  f"{epoch_time:.1f}s/epoch ({sec_per_batch:.2f}s/batch)  "
                  f"elapsed={elapsed / 60:.1f}min  "
                  f"ETA this fold={eta_this_fold / 60:.1f}min  "
                  f"ETA all folds={eta_total / 60:.1f}min")

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= BASELINE_PATIENCE:
                print(f"    early stopping at epoch {epoch + 1} (no improvement for {BASELINE_PATIENCE} epochs)")
                break

        model.load_state_dict(best_state)
        print(f"Fold {fold_idx + 1} best val_loss: {best_val_loss:.4f} (model restored)")

        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                vol = batch["volume"].to(device)
                logits = model(vol).cpu()
                for i, pid in enumerate(batch["PatientID"]):
                    oof_logits[pid] = logits[i].numpy()
                    oof_labels[pid] = {"idh": batch["label"][i].item()}

    all_pids = list(oof_labels.keys())
    oof_valid = {"idh": {p: True for p in all_pids}}
    r = summarize_task("idh", 2, oof_labels, {"idh": oof_logits}, oof_valid, all_pids)
    record_oof_predictions(condition_name, "idh", 2, oof_labels, {"idh": oof_logits}, oof_valid, all_pids)
    if r is None:
        print("  Insufficient valid samples.")
        return []
    r["condition"] = condition_name
    print(f"  IDH      n={r['n']:4d}  AUC={r['auc']:.3f}  "
          f"Acc={r['accuracy']:.3f}  BalAcc={r['balanced_accuracy']:.3f}")
    return [r]


# ======================================================================
# MASTER RUN
# ======================================================================
def main():
    print("\n" + "=" * 70 + "\nPHASE 5 (PATCHED): Baseline and Ablation Comparisons\n" + "=" * 70)

    master_df, clinical_cols, cols_to_normalize = build_master_df()
    folds = get_or_create_folds(master_df)
    print(f"\nUsing {N_FOLDS} fixed folds across all conditions "
          f"({len(master_df)} patients total)")

    all_results = []
    all_results += run_baseline_condition(master_df, folds)

    for t in TASKS:
        all_results += run_mil_condition(master_df, clinical_cols, cols_to_normalize, folds,
                                          tasks={t: TASKS[t]}, use_fusion=False,
                                          condition_name=f"(ii) 2.5D MIL single-task [{t}]")

    all_results += run_mil_condition(master_df, clinical_cols, cols_to_normalize, folds, tasks=TASKS,
                                      use_fusion=False, condition_name="(iii) 2.5D MIL multi-task")

    all_results += run_mil_condition(master_df, clinical_cols, cols_to_normalize, folds, tasks=TASKS,
                                      use_fusion=True,
                                      condition_name="(iv) 2.5D MIL multi-task + fusion")

    # PATCH A: new radiomics/clinical-only condition, same folds, same tasks.
    all_results += run_structured_only_condition(master_df, clinical_cols, cols_to_normalize,
                                                  folds, tasks=TASKS)

    df = pd.DataFrame(all_results)
    df_display = df[["condition", "task", "n", "accuracy", "balanced_accuracy",
                      "macro_f1", "auc", "auc_ci_low", "auc_ci_high"]]
    df_display.to_csv(RESULTS_CSV, index=False)

    # PATCH B: flush all per-patient OOF predictions collected across every
    # condition/task to disk. This is the file phase5_significance_test.py
    # (companion script) consumes to run DeLong / permutation tests.
    oof_df = pd.DataFrame(ALL_OOF_ROWS)
    oof_df.to_csv(OOF_CSV, index=False)
    print(f"\nSaved {len(oof_df)} per-patient OOF prediction rows to {OOF_CSV}")

    print("\n" + "=" * 70 + "\nPHASE 5 SUMMARY -- pooled out-of-fold, all conditions\n" + "=" * 70)
    with pd.option_context("display.max_rows", None, "display.width", 140):
        print(df_display.round(3).to_string(index=False))
    print(f"\nSaved full results (incl. confusion matrices) to {RESULTS_CSV}")
    print("\nDONE")
    return df


if __name__ == "__main__":
    main()

In [ ]:
import os
for f in ["phase5_ablation_results.csv", "phase5_oof_predictions.csv", "phase5_fold_splits.json"]:
    p = f"/kaggle/working/{f}"
    print(f, "EXISTS" if os.path.exists(p) else "MISSING",
          os.path.getmtime(p) if os.path.exists(p) else "")